### Tested with Same Corpas
GSM8K 1870 sample
Adapter : Qwen + GSM8K  

#### Part 01

# Notebook 6 -- Three-Angle Evaluation on SVAMP
## SLM-to-SLM Guided Reasoning Pipeline

**Why SVAMP instead of GSM8K?**

SVAMP (Simple Variations on Arithmetic Math Problems) is specifically designed
to break solvers that rely on surface patterns rather than real understanding.
The dataset takes GSM8K-style problems and applies small structural changes
(swapping what is asked, changing relations) -- a model that "guesses" from
keywords will fail. This means the guidance gap should be *larger and more
meaningful* than on GSM8K.

| Property | GSM8K | SVAMP |
|---|---|---|
| Size (test) | 1,319 | 1,000 |
| Difficulty | Grade school | Robustness-focused |
| Pattern-gaming risk | High | Low (by design) |
| HuggingFace ID | `openai/gsm8k` | `ChilleD/SVAMP` |

**Pipeline (unchanged from Notebook 5)**
```
Question --> Fine-tuned Qwen 3B (Guide) --> Plan --> Qwen 1.5B (Solver) x5 --> Vote --> Answer
Baseline : Question -----------------------------------------> Qwen 1.5B (Solver) x5 --> Vote
```

**Improvements in this notebook**
- Fixed random seed applied ONCE before sampling -- both modes use identical questions
- Richer answer extractor handles floats, LaTeX, bold markdown
- Per-question wasted-vote tracking (not just aggregate)
- Refiner effectiveness logged separately
- Angle 2 adds per-question win/loss/tie comparison
- Angle 3 adds explicit false-confidence count per bucket
- Summary table uses actual numeric cells (not f-string boxes)


In [ ]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")


In [ ]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print('✅ HuggingFace login done')


In [ ]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/svamp_eval"
ADAPTER_PATH = "/kaggle/input/svamp-adapter/adapter_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")
print(f"Adapter : {ADAPTER_PATH}")

In [ ]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "ChilleD/SVAMP",   # huggingface dataset id
    "dataset_split"       : "test",
    "max_eval_samples"    : 400,   # SVAMP test has 1000; increase to 1000 for full run
    "random_seed"         : 42,    # FIXED -- same seed for BOTH guided and baseline

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.7,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 350,

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


In [ ]:
# CELL 5 -- Load SVAMP dataset
# SVAMP fields: Body, Question, Equation, Answer (numeric)
# We combine Body + Question into a single question string.

print("Loading SVAMP from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits     : {list(raw_ds.keys())}")
print(f"Features   : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Test size  : {len(raw_ds[CONFIG['dataset_split']])}")
print(f"Example    :")
ex = raw_ds[CONFIG["dataset_split"]][0]
for k, v in ex.items():
    print(f"  {k}: {v}")


def normalise_svamp(item):
    """Convert SVAMP record to {question, answer} used by the pipeline."""
    q = item["Body"].strip().rstrip(".") + " " + item["Question"].strip()
    ans = item["Answer"]
    # Store as clean integer string when possible (5.0 -> "5")
    if isinstance(ans, float) and ans == int(ans):
        ans_str = str(int(ans))
    else:
        ans_str = str(ans)
    return {"question": q, "answer": ans_str}


all_data = [normalise_svamp(x) for x in raw_ds[CONFIG["dataset_split"]]]

# ---- CRITICAL: set seed ONCE here, before any sampling ----------
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

# Fingerprint so we can verify reproducibility
print(f"First Q  : {test_data[0]['question'][:80]}...")
print(f"First A  : {test_data[0]['answer']}")
print(f"Last  Q  : {test_data[-1]['question'][:60]}...")
print("SVAMP loaded")


In [ ]:
# CELL 6 -- Answer extraction (upgraded for SVAMP)
# SVAMP answers are clean integers or simple floats -- no dollar signs or commas.
# We still need to handle all Qwen output styles.

def normalise_num(s):
    """Convert numeric string to canonical form. 5.0 -> '5', 3.14 -> '3.14'."""
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError:
        return s


def extract_gt_answer(answer_str):
    """SVAMP GT is already clean -- just normalise."""
    return normalise_num(str(answer_str))


def extract_pred_answer(text):
    """
    Multi-pattern extractor. Returns empty string on failure
    (never falls back to a random number from the text).

    Priority:
      1. #### N          -- standard format we request
      2. \\boxed{N}      -- Qwen's preferred LaTeX style
      3. 'the answer is' -- common phrasing
      4. '= N' at end of line
      5. **N** at end    -- bold markdown
      6. 'therefore N'   -- conclusion phrases
    """
    # 1
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    # 2
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    # 3
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    # 4
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    # 5
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    # 6
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""


# --- Self-test ---
_tests = [
    ("#### 42",                 "42"),
    ("#### 3.5",                "3.5"),
    ("\\boxed{100}",            "100"),
    ("The answer is 7",         "7"),
    ("Total = 20",              "20"),
    ("**200**.",                "200"),
    ("Therefore, 13",           "13"),
    ("Some unrelated text",     ""),
]
ok = True
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    status = "OK" if got == exp else "FAIL"
    if got != exp: ok = False
    print(f"  {status}  '{txt[:35]}' -> '{got}' (expected '{exp}')")
print("All extractor tests passed" if ok else "EXTRACTOR HAS FAILURES -- fix before running eval")


In [ ]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        ADAPTER_PATH,
        "/kaggle/input/datasets/sufiantabdullah/final-adapter",
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token










guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")
    print("Results will still run but guide quality may be lower.")

guide_model.eval()
print(f"Guide VRAM : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models) : {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                 : {headroom:.1f} GB")
if headroom < 2:
    print("WARNING: Very tight. Reduce n_votes to 3 if you see OOM errors.")
else:
    print("Memory OK")


In [ ]:
# CELL 9 -- Prompts and generation functions

GUIDE_SYSTEM = (
    "You are a math problem decomposition assistant.\n"
    "Identify ONLY the arithmetic operations needed. 1-3 steps maximum.\n"
    "State WHAT is being compared or combined using EXACT numbers.\n"
    "Do NOT invent steps. Do NOT reorder the question.\n\n"
    "BAD:  Step 1: Calculate remaining cookies.\n"
    "GOOD: Step 1: Eaten - Given = 14 - 13 = ?\n"
    "      Step 2: Answer = 14 - 13\n\n"
    "If the question asks 'how many MORE did X than Y', "
    "the operation is X - Y, not Y - X.\n"
    "No final answer number. Just the operation steps."
)

SOLVE_SYSTEM = (
    "You are a math problem solver.\n"
    "Compute each step numerically. No markdown. No bullet points. No headers.\n"
    "Write plain arithmetic steps only.\n"
    "Your absolute last line must be: #### [number]\n"
    "NEVER write ### or ** in your response.\n\n"
    "Example:\n"
    "Eaten = 14. Given = 13.\n"
    "Difference = 14 - 13 = 1.\n"
    "#### 1"
)

BASELINE_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Read the problem carefully. Solve step by step, showing every calculation.\n"
    "Your FINAL line must be exactly: #### [number]"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts on this problem gave different answers.\n"
    "Re-solve completely from scratch using a fresh approach.\n"
    "Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)


def run_qwen(mdl, tok, messages, max_tokens, temperature):
    """Call any Qwen-family model and return generated text."""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(mdl.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_plan(question):
    return run_qwen(
        guide_model, guide_tok,
        [{"role": "system", "content": GUIDE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = 350,
        temperature = CONFIG["guide_temperature"],
    )


def generate_guided(question, plan):
    content = f"Problem: {question}\n\nPlan (follow each step):\n{plan}\n\nSolve step by step:"
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": SOLVE_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_refiner(question, plan, candidates):
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem: {question}\n\n"
        f"Plan:\n{plan}\n\n"
        f"Previous attempts disagreed: {cands}\n"
        "Re-solve carefully from scratch:"
    )
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )


print("Generation functions ready")


In [ ]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, plan, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Returns a dict with all metrics needed for the three angles.

    Fields:
      final_answer     : chosen answer string
      strategy         : 'majority' | 'refiner_tiebreak' | 'coin_flip'
      confidence       : top_count / total_votes
      correct_votes    : votes that matched gt_answer
      vote_consistency : correct_votes / total_votes
      wasted_votes     : votes that did NOT match final_answer
      refiner_used     : bool
      refiner_correct  : bool or None
    """
    valid = [a for a in answers if a and a.strip()]
    if not valid:
        valid = answers  # fallback — keep all if everything failed
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(answers)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / total

    is_majority = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        # Tie: call refiner as an extra vote
        ref_raw    = generate_refiner(question, plan, list(answers))
        ref_ans    = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_votes  = answers + [ref_ans]
        new_counts = Counter(all_votes)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_votes), 4)
        vote_counts = new_counts
        total      = len(all_votes)
        correct_votes    = new_counts.get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / total
        wasted           = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "correct_votes"    : correct_votes,
        "total_votes"      : total,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }


print("Voting logic ready")
print("  majority        -> clear winner across votes")
print("  refiner_tiebreak-> tie broken by refiner call")
print("  coin_flip       -> still tied after refiner")


In [ ]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (SVAMP)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question : {q}")
print(f"GT Answer: {gt}")

# Guided
print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

g = vote_and_decide(guided_votes, q, plan, gt)
print(f"\n  Result   : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")

# Baseline
print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'")

b = vote_and_decide(base_votes, q, "baseline", gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


In [ ]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided  (guide plan + solver x5)
#   Mode B: Baseline (solver x5, no plan)
#
# Both modes share the exact same question list.
# All per-question metrics are embedded directly in each result record.

print(f"Dual evaluation: {len(test_data)} SVAMP questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

all_results  = []   # guided results
base_results = []   # baseline results
start_idx    = 0

# Resume from checkpoint
if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="SVAMP Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -------------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, plan, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "correct_votes"    : g_dec["correct_votes"],
            "total_votes"      : g_dec["total_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
            "vote_counts"      : g_dec["vote_counts"],
            "plan"             : plan,
        })
    except RuntimeError as e:
        all_results.append({
            "mode": "guided", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # ---- BASELINE -----------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, "baseline", gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "correct_votes"    : b_dec["correct_votes"],
            "total_votes"      : b_dec["total_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : None,
            "vote_counts"      : b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # Checkpoint
    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] Guided: {g_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(CONFIG["results_file"], "w") as f:
    for r in all_results + base_results:
        f.write(json.dumps(r) + "\n")

g_c = sum(r["correct"] for r in all_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nEvaluation complete.")
print(f"  Guided   : {g_c}/{len(all_results)} = {g_c/len(all_results)*100:.1f}%")
print(f"  Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"  Delta    : +{(g_c/len(all_results) - b_c/len(base_results))*100:.1f} percentage points")
   

In [ ]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
# =================================================================
# We compare three setups by accuracy and compute cost:
#   Baseline : 1.5B solver x5 votes              = 7.5B param-passes
#   Guided   : 3B guide x1 + 1.5B solver x5      = 10.5B param-passes
#   Upper    : 3B model x5 (hypothetical ceiling) = 15.0B param-passes
#
# Key question: does the guided setup earn more accuracy per
# extra param-pass it spends compared to the baseline?
# =================================================================

G = CONFIG["guide_params_B"]    # 3.0
S = CONFIG["solver_params_B"]   # 1.5
N = CONFIG["n_votes"]           # 5

guided_compute   = (G * 1) + (S * N)    # 3 + 7.5 = 10.5
baseline_compute = S * N                 # 7.5
upper_compute    = G * N                 # 15.0

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

# Accuracy-per-billion-param-passes
g_eff = g_acc / guided_compute
b_eff = b_acc / baseline_compute

# Wasted votes: votes that did not match final answer
g_wasted       = sum(r["wasted_votes"] for r in all_results)
b_wasted       = sum(r["wasted_votes"] for r in base_results)
total_possible = len(all_results) * N

# Refiner stats
ref_triggered  = sum(r["refiner_used"] for r in all_results)
ref_correct    = sum(1 for r in all_results
                     if r["refiner_used"] and r.get("refiner_correct"))

# Per-strategy accuracy
strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats:
        strategy_stats[s] = {"n": 0, "correct": 0}
    strategy_stats[s]["n"] += 1
    if r["correct"]:
        strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (SVAMP)")
print("=" * 65)
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9} | {'Acc/B':>7}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}-+-{'-'*7}")
print(f"  {'Baseline (1.5B x ' + str(N) + ')':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}% | {b_eff:>6.3f}")
print(f"  {'Guided  (3B x1 + 1.5B x' + str(N) + ')':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}% | {g_eff:>6.3f}")
print(f"  {'Upper   (3B x ' + str(N) + ')':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9} |")

savings_pct = (1 - guided_compute / upper_compute) * 100
acc_gain    = g_acc - b_acc
print(f"\n  Accuracy gain over baseline  : +{acc_gain:.1f} percentage points")
print(f"  Compute savings vs upper     : {savings_pct:.0f}% cheaper")
print(f"  Efficiency lift (acc/B)      : {g_eff:.3f} vs {b_eff:.3f} (guided)")

print(f"\n  Wasted votes (votes != final answer):")
print(f"    Guided   : {g_wasted} / {total_possible}  ({g_wasted/total_possible*100:.1f}%)")
print(f"    Baseline : {b_wasted} / {total_possible}  ({b_wasted/total_possible*100:.1f}%)")
print(f"    Saved    : {b_wasted - g_wasted} fewer wasted compute passes with guidance")

if ref_triggered > 0:
    print(f"\n  Refiner (tie-breaker) stats:")
    print(f"    Triggered : {ref_triggered} / {len(all_results)} questions")
    print(f"    Correct   : {ref_correct} / {ref_triggered}  ({ref_correct/ref_triggered*100:.1f}% of ties resolved correctly)")

print(f"\n  Decision strategy breakdown (guided):")
print(f"  {'Strategy':<22} | {'Count':>6} | {'Accuracy':>9}")
print(f"  {'-'*22}-+-{'-'*6}-+-{'-'*9}")
for s, v in sorted(strategy_stats.items(), key=lambda x: -x[1]["n"]):
    acc_s = v["correct"] / v["n"] * 100 if v["n"] else 0
    print(f"  {s:<22} | {v['n']:>6} | {acc_s:>8.1f}%")

angle1 = {
    "dataset"               : "SVAMP",
    "n_questions"           : len(all_results),
    "guided_compute_B"      : guided_compute,
    "baseline_compute_B"    : baseline_compute,
    "upper_compute_B"       : upper_compute,
    "guided_accuracy"       : round(g_acc, 2),
    "baseline_accuracy"     : round(b_acc, 2),
    "accuracy_gain"         : round(acc_gain, 2),
    "compute_savings_pct"   : round(savings_pct, 1),
    "guided_efficiency"     : round(g_eff, 4),
    "baseline_efficiency"   : round(b_eff, 4),
    "guided_wasted_votes"   : g_wasted,
    "baseline_wasted_votes" : b_wasted,
    "wasted_votes_saved"    : b_wasted - g_wasted,
    "refiner_triggered"     : ref_triggered,
    "refiner_correct"       : ref_correct,
    "strategy_breakdown"    : strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")


In [ ]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
# =================================================================
# Vote consistency = fraction of votes (out of 5) that matched GT.
# A high-consistency question means the model reliably solves it.
# A low-consistency question means it got lucky on the final vote.
#
# If guidance works, we expect BOTH:
#   - Higher mean consistency (more votes correct per question)
#   - More questions in the "high" bucket (4-5 correct votes)
#
# Per-question comparison: on each question, does guided produce
# more correct votes than baseline? Win/Loss/Tie.
# =================================================================

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

# Per-question: guided better / baseline better / tied
guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

# Distribution buckets
def bucket(scores):
    return {
        "all_wrong  (0%)": sum(1 for s in scores if s == 0.0),
        "low       (1-39%)": sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

# For questions where the pipeline was CORRECT, how consistent were the votes?
# (high consistency + correct = true reliable solving, not a lucky majority)
g_corr_cons = [r["vote_consistency"] for r in all_results  if r["correct"]]
b_corr_cons = [r["vote_consistency"] for r in base_results if r["correct"]]

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (SVAMP)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes per question):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Lift     : {lift:.2f}x  (guided produces {lift:.1f}x more correct votes per question)")

print(f"\n  Per-question comparison (same questions, both modes):")
print(f"    Guided beats baseline : {guided_wins} / {len(all_results)} questions")
print(f"    Baseline beats guided : {baseline_wins} / {len(all_results)} questions")
print(f"    Equal                 : {tied} / {len(all_results)} questions")

print(f"\n  Distribution of vote consistency:")
print(f"  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gv  = g_dist[bkt]
    bv  = b_dist[bkt]
    dif = gv - bv
    sign = "+" if dif >= 0 else ""
    print(f"  {bkt:<22} | {gv:>8} | {bv:>8} | {sign+str(dif):>6}")

if g_corr_cons:
    print(f"\n  Among CORRECT questions only -- average vote consistency:")
    print(f"    Guided   : {np.mean(g_corr_cons)*100:.1f}%  (n={len(g_corr_cons)})")
    print(f"    Baseline : {np.mean(b_corr_cons)*100:.1f}%  (n={len(b_corr_cons)})")
    print("    (High consistency + correct = genuine reliable solving, not lucky vote)")

angle2 = {
    "dataset"                   : "SVAMP",
    "n_questions"               : len(all_results),
    "guided_mean_consistency"   : round(g_mean, 4),
    "baseline_mean_consistency" : round(b_mean, 4),
    "consistency_lift"          : round(lift, 4),
    "guided_wins"               : guided_wins,
    "baseline_wins"             : baseline_wins,
    "tied"                      : tied,
    "guided_distribution"       : g_dist,
    "baseline_distribution"     : b_dist,
    "guided_correct_q_consistency"   : round(np.mean(g_corr_cons), 4) if g_corr_cons else 0,
    "baseline_correct_q_consistency" : round(np.mean(b_corr_cons), 4) if b_corr_cons else 0,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")


In [ ]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
# =================================================================
# Confidence = fraction of votes that agreed on the winning answer.
# Perfect calibration: "80% confident" means 80% accurate.
#
# ECE (Expected Calibration Error) measures the average gap
# between stated confidence and actual accuracy across all buckets.
# LOWER ECE = more trustworthy confidence signal.
#
# False confidence = all 5 votes agree on the WRONG answer.
# This is the most dangerous failure: the system is maximally
# confident and maximally wrong simultaneously.
#
# Guidance should suppress false confidence by steering votes
# toward correct reasoning paths rather than shared mistakes.
# =================================================================

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40,  0.25),
    ]
    n_total   = len(results)
    ece       = 0.0
    calib_out = []

    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")

    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({
            "bucket": name, "count": n,
            "accuracy": round(acc, 4), "expected": mid, "gap": round(gap, 4)
        })

    hc_items = [r for r in results if r["confidence"] >= 0.80]
    hc_acc   = sum(r["correct"] for r in hc_items) / max(1, len(hc_items)) * 100
    print(f"  {'ECE (lower=better)':<26}   {ece:>5.4f}")
    print(f"  High-conf questions : {len(hc_items)}  |  Accuracy when confident: {hc_acc:.1f}%")
    print(f"  Confidently WRONG   : {false_conf} questions (false confidence)")
    return ece, calib_out, false_conf


print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (SVAMP)")
print("=" * 65)
print("Ideal: accuracy at each confidence level matches that level.")
print("False confidence: model agrees on wrong answer with full certainty.")

g_ece, g_calib, g_false = calibration_report(all_results,  "GUIDED pipeline")
b_ece, b_calib, b_false = calibration_report(base_results, "BASELINE (no plan)")

improve_pct = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE Summary:")
print(f"    Guided   ECE : {g_ece:.4f}")
print(f"    Baseline ECE : {b_ece:.4f}")
print(f"    Improvement  : {improve_pct:.1f}% better calibrated")
print(f"\n  False Confidence (confident AND wrong):")
print(f"    Guided   : {g_false} questions")
print(f"    Baseline : {b_false} questions")
print(f"    Reduction: {b_false - g_false} fewer false-confidence questions with guidance")

angle3 = {
    "dataset"                 : "SVAMP",
    "n_questions"             : len(all_results),
    "guided_ece"              : round(g_ece, 4),
    "baseline_ece"            : round(b_ece, 4),
    "ece_improvement_pct"     : round(improve_pct, 2),
    "guided_false_confidence" : g_false,
    "baseline_false_confidence": b_false,
    "false_conf_reduction"    : b_false - g_false,
    "guided_calibration"      : g_calib,
    "baseline_calibration"    : b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")


In [ ]:
# CELL 16 -- Full Paper Summary (all three angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n  = a1["n_questions"]
ga = a1["guided_accuracy"]
ba = a1["baseline_accuracy"]
gc = a1["guided_compute_B"]
uc = a1["upper_compute_B"]

print("=" * 68)
print("  SVAMP EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset  : SVAMP  |  Questions: {n}  |  Seed: {CONFIG['random_seed']}")
print(f"  Models   : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print()

rows = [
    ["Metric",                    "Baseline",       "Guided",          "Change"],
    ["Overall Accuracy",
     str(ba) + "%",               str(ga) + "%",
     "+" + str(round(ga-ba,1)) + " pts"],
    ["Compute Cost",
     str(a1['baseline_compute_B']) + "B param-passes",
     str(gc) + "B param-passes",
     str(a1['compute_savings_pct']) + "% cheaper than ceiling"],
    ["Wasted Votes",
     str(a1['baseline_wasted_votes']),
     str(a1['guided_wasted_votes']),
     str(a1['wasted_votes_saved']) + " fewer wasted passes"],
    ["Vote Consistency",
     str(round(a2['baseline_mean_consistency']*100,1)) + "%",
     str(round(a2['guided_mean_consistency']*100,1)) + "%",
     str(round(a2['consistency_lift'],2)) + "x lift"],
    ["High-Agreement Questions",
     str(a2['baseline_distribution']['high   (80-100%)']),
     str(a2['guided_distribution']['high   (80-100%)']),
     ""],
    ["Guided Wins Per-Question",
     "--",
     str(a2['guided_wins']) + " / " + str(n),
     ""],
    ["ECE (lower = better)",
     str(a3['baseline_ece']),
     str(a3['guided_ece']),
     str(a3['ece_improvement_pct']) + "% better"],
    ["False Confidence Count",
     str(a3['baseline_false_confidence']),
     str(a3['guided_false_confidence']),
     str(a3['false_conf_reduction']) + " fewer"],
]

col_w = [28, 18, 18, 28]
sep   = "-+-".join("-" * w for w in col_w)
for i, row in enumerate(rows):
    line = " | ".join(str(cell).ljust(col_w[j]) for j, cell in enumerate(row))
    print("  " + line)
    if i == 0:
        print("  " + sep)

if a1.get("refiner_triggered", 0) > 0:
    rt = a1["refiner_triggered"]
    rc = a1.get("refiner_correct", 0)
    print(f"\n  Refiner: triggered {rt} times, resolved {rc} correctly ({rc/rt*100:.1f}%)")

# Save full report
full = {
    "dataset": "SVAMP", "seed": CONFIG["random_seed"],
    "n_questions": n, "angle1": a1, "angle2": a2, "angle3": a3,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(full, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}")
print("Commit this notebook to preserve outputs.")


### Part 2

# Notebook 9 -- Three-Angle Evaluation on ASDiv
## SLM-to-SLM Guided Reasoning Pipeline

**Why ASDiv?**

ASDiv (Academic Subtypes Diversity) tags every problem with its operation type:
Addition, Subtraction, Multiplication, Division, and multi-step combinations.
This lets us break results down by category and ask:
*"Does guidance help more on division than addition?"*
That is a much richer finding than a single accuracy number across all problems.

| Property | SVAMP | ASDiv |
|---|---|---|
| HuggingFace ID | `ChilleD/SVAMP` | `EleutherAI/asdiv` |
| Size | 1,000 | ~2,300 |
| Format | Open numeric | Open numeric |
| Key field | None | `solution_type` -- operation category |
| Difficulty | Robustness-focused | Wider operation variety |

**Key addition vs SVAMP notebook**
- Per operation-type accuracy breakdown (Addition / Subtraction / Multiplication / Division / multi-step)
- Guidance lift measured separately per operation type
- Vote consistency broken down per operation type
- This enables the paper claim: "guidance helps most on X-type problems"

**Pipeline (identical structure)**
```
Question --> Fine-tuned Qwen 3B (Guide) --> Plan --> Qwen 1.5B (Solver) x5 --> Vote --> Answer
Baseline: Question -----------------------------------------> Qwen 1.5B x5 --> Vote
```


In [ ]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "EleutherAI/asdiv",
    "dataset_split"       : "validation",   # ASDiv has train/validation splits
    "max_eval_samples"    : 300,            # validation has ~2300; increase for full run
    "random_seed"         : 42,             # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.7,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 350,

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "angle4_file"         : f"{OUTPUT_DIR}/angle4_by_operation_type.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


In [ ]:
# CELL 5 -- Load ASDiv dataset
# ASDiv fields: body, question, solution_type, answer, formula
#
# solution_type examples:
#   "Addition", "Subtraction", "Multiplication", "Division",
#   "Common-Division", "Comparison", "Sum", "Difference" etc.
#
# We combine body + question into a single question string,
# extract the numeric answer, and map solution_type to a
# broad 5-category label used for the per-type analysis.

print("Loading ASDiv from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Features : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Split size: {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record:")
for k, v in ex.items():
    print(f"  {k}: {v}")


# Map the fine-grained solution_type into 5 broad categories
OP_MAP = {
    "addition"         : "Addition",
    "sum"              : "Addition",
    "subtraction"      : "Subtraction",
    "difference"       : "Subtraction",
    "comparison"       : "Subtraction",
    "multiplication"   : "Multiplication",
    "division"         : "Division",
    "common-division"  : "Division",
    "floor-division"   : "Division",
}

def broad_op(solution_type):
    """Map fine-grained solution_type to one of 5 broad categories."""
    st = str(solution_type).lower().strip()
    for key, val in OP_MAP.items():
        if key in st:
            return val
    # multi-step or unknown
    return "Multi-step"


def normalise_asdiv(item):
    """Convert ASDiv record to {question, answer, op_type} pipeline format."""
    q = item["body"].strip().rstrip(".") + " " + item["question"].strip()

    # Answer may be "10 cookies" or "10" -- keep only the leading numeric part
    raw_ans = str(item["answer"]).strip().replace(",", "")
    m = re.match(r"(-?[\d\.]+)", raw_ans)
    ans_str = m.group(1) if m else raw_ans

    # Normalise float: 5.0 -> "5"
    try:
        f = float(ans_str)
        ans_str = str(int(f)) if f == int(f) else str(round(f, 4))
    except Exception:
        pass

    op_type      = broad_op(item.get("solution_type", ""))
    solution_type = str(item.get("solution_type", ""))

    return {
        "question"      : q,
        "answer"        : ans_str,
        "op_type"       : op_type,
        "solution_type" : solution_type,
    }


all_data = [normalise_asdiv(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Show operation type distribution before sampling
op_counts = Counter(d["op_type"] for d in all_data)
print(f"\nOperation type distribution ({len(all_data)} total):")
for op, cnt in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")

# CRITICAL: fix seed ONCE here, before any sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

# Show op distribution in sampled set
sampled_op = Counter(d["op_type"] for d in test_data)
print(f"\nSampled operation distribution:")
for op, cnt in sorted(sampled_op.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")

print(f"\nFirst Q : {test_data[0]['question'][:80]}...")
print(f"First A : {test_data[0]['answer']}  |  op: {test_data[0]['op_type']}")
print("ASDiv loaded")


In [ ]:
# CELL 6 -- Answer extraction (same as SVAMP -- numeric answers)
# ASDiv answers are integers or simple floats.
# Answers sometimes include units like "10 cookies" -- we strip those in Cell 5.

def normalise_num(s):
    """Canonical numeric string. 5.0 -> '5', 3.14 -> '3.14'."""
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError:
        return s


def extract_gt_answer(answer_str):
    """ASDiv GT is already cleaned in normalise_asdiv -- just normalise."""
    return normalise_num(str(answer_str))


def extract_pred_answer(text):
    """
    Multi-pattern extractor. Returns empty string on failure.
    Never falls back to a random number from the text.

    Priority:
      1. #### N          -- standard format we request
      2. \\boxed{N}      -- Qwen's preferred LaTeX style
      3. 'the answer is' -- common phrasing
      4. '= N' at end of line
      5. **N** at end    -- bold markdown
      6. 'therefore N'   -- conclusion phrases
    """
    # 1
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    # 2
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    # 3
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    # 4
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    # 5
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    # 6
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""


# --- Self-test ---
_tests = [
    ("#### 42",             "42"),
    ("#### 3.5",            "3.5"),
    ("\\boxed{100}",        "100"),
    ("The answer is 7",     "7"),
    ("Total = 20",          "20"),
    ("**200**.",            "200"),
    ("Therefore, 13",       "13"),
    ("Some unrelated text", ""),
]
ok = True
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    status = "OK" if got == exp else "FAIL"
    if got != exp: ok = False
    print(f"  {status}  '{txt[:35]}' -> '{got}' (expected '{exp}')")
print("\nAll extractor tests passed" if ok else "\nEXTRACTOR HAS FAILURES -- fix before running eval")


In [ ]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        ADAPTER_PATH,
        "/kaggle/input/datasets/sufiantabdullah/final-adapter",
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token



if 'guide_model' in dir():
    del guide_model
if 'adapter_path' in dir():
    del adapter_path
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Loading guide base: {CONFIG['guide_base']}")



guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


In [ ]:
# CELL 9 -- Prompts and generation functions

GUIDE_SYSTEM = (
    "You are a math problem decomposition assistant.\n"
    "Identify ONLY the arithmetic operations needed. 1-3 steps maximum.\n"
    "State WHAT is being compared or combined using EXACT numbers.\n"
    "Do NOT invent steps. Do NOT reorder the question.\n\n"
    "BAD:  Step 1: Calculate remaining cookies.\n"
    "GOOD: Step 1: Eaten - Given = 14 - 13 = ?\n"
    "      Step 2: Answer = 14 - 13\n\n"
    "If the question asks 'how many MORE did X than Y', "
    "the operation is X - Y, not Y - X.\n"
    "No final answer number. Just the operation steps with actual numbers."
)

SOLVE_SYSTEM = (
    "You are a math problem solver.\n"
    "Compute each step numerically. No markdown. No bullet points. No headers.\n"
    "Write plain arithmetic steps only.\n"
    "Your absolute last line must be: #### [number]\n"
    "NEVER write ### or ** in your response.\n\n"
    "Example:\n"
    "Eaten = 14. Given = 13.\n"
    "Difference = 14 - 13 = 1.\n"
    "#### 1"
)

BASELINE_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Read the problem carefully. Solve step by step, showing every calculation.\n"
    "Your FINAL line must be exactly: #### [number]"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts on this problem gave different answers.\n"
    "Ignore all previous attempts. Re-solve completely from scratch.\n"
    "Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)


def run_qwen(mdl, tok, messages, max_tokens, temperature):
    """Call any Qwen-family model and return generated text."""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(mdl.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_plan(question):
    return run_qwen(
        guide_model, guide_tok,
        [{"role": "system", "content": GUIDE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = 350,
        temperature = CONFIG["guide_temperature"],
    )


def generate_guided(question, plan):
    content = f"Problem: {question}\n\nPlan (follow each step):\n{plan}\n\nSolve step by step:"
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": SOLVE_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_refiner(question, candidates):
    # Refiner runs WITHOUT the plan -- plan may have caused the tie
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem: {question}\n\n"
        f"Previous attempts gave: {cands}\n"
        "Ignore all previous attempts. Solve from scratch:"
    )
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )


print("Generation functions ready")
print("  Refiner runs WITHOUT plan (avoids inheriting bad plan)")


In [ ]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Filters empty responses before counting.

    Returns dict with all metrics needed for the three angles.
    """
    valid = [a for a in answers if a and a.strip()]
    if not valid:
        valid = answers  # fallback -- keep all if extraction completely failed

    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        # Tie: refiner runs WITHOUT plan
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted           = total - new_top_c
        vote_counts      = new_counts

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "correct_votes"    : correct_votes,
        "total_votes"      : total,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }


print("Voting logic ready")
print("  majority         -> clear winner across votes")
print("  refiner_tiebreak -> tie broken by refiner (no plan)")
print("  coin_flip        -> still tied after refiner")


In [ ]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (ASDiv)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
op   = item["op_type"]
print(f"Question : {q}")
print(f"GT Answer: {gt}  |  Operation type: {op}")

# Guided
print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")

# Baseline
print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


In [ ]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided  (guide plan + solver x5)
#   Mode B: Baseline (solver x5, no plan)
#
# op_type is stored in every record so Angle 4 can group by operation.

print(f"Dual evaluation: {len(test_data)} ASDiv questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="ASDiv Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])
    op_type   = item["op_type"]

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "op_type"          : op_type,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "correct_votes"    : g_dec["correct_votes"],
            "total_votes"      : g_dec["total_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
            "vote_counts"      : g_dec["vote_counts"],
            "plan"             : plan,
        })
    except RuntimeError as e:
        all_results.append({
            "mode": "guided", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "op_type"          : op_type,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "correct_votes"    : b_dec["correct_votes"],
            "total_votes"      : b_dec["total_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : None,
            "vote_counts"      : b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] Guided: {g_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

with open(CONFIG["results_file"], "w") as f:
    for r in all_results + base_results:
        f.write(json.dumps(r) + "\n")

g_c = sum(r["correct"] for r in all_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nEvaluation complete.")
print(f"  Guided   : {g_c}/{len(all_results)} = {g_c/len(all_results)*100:.1f}%")
print(f"  Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"  Delta    : +{(g_c/len(all_results) - b_c/len(base_results))*100:.1f} percentage points")  

In [ ]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]

guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

g_eff       = g_acc / guided_compute
b_eff       = b_acc / baseline_compute
savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted       = sum(r["wasted_votes"] for r in all_results)
b_wasted       = sum(r["wasted_votes"] for r in base_results)
total_possible = len(all_results) * N

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (ASDiv)")
print("=" * 65)
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9} | {'Acc/B':>7}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}-+-{'-'*7}")
print(f"  {'Baseline (1.5B x ' + str(N) + ')':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}% | {b_eff:>6.3f}")
print(f"  {'Guided  (3B x1 + 1.5B x' + str(N) + ')':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}% | {g_eff:>6.3f}")
print(f"  {'Upper   (3B x ' + str(N) + ')':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9} |")
print(f"\n  Accuracy gain over baseline  : +{g_acc - b_acc:.1f} percentage points")
print(f"  Compute savings vs upper     : {savings_pct:.0f}% cheaper")
print(f"  Wasted votes saved           : {b_wasted - g_wasted}  ({g_wasted} guided vs {b_wasted} baseline)")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")
print(f"\n  Strategy breakdown (guided):")
for s, v in sorted(strategy_stats.items(), key=lambda x: -x[1]["n"]):
    acc_s = v["correct"]/v["n"]*100 if v["n"] else 0
    print(f"    {s:<22}: {v['n']:>4} questions  {acc_s:>6.1f}% accuracy")

angle1 = {
    "dataset": "ASDiv", "n_questions": len(all_results),
    "guided_compute_B": guided_compute, "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute, "guided_accuracy": round(g_acc,2),
    "baseline_accuracy": round(b_acc,2), "accuracy_gain": round(g_acc-b_acc,2),
    "compute_savings_pct": round(savings_pct,1),
    "guided_efficiency": round(g_eff,4), "baseline_efficiency": round(b_eff,4),
    "guided_wasted_votes": g_wasted, "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted-g_wasted,
    "refiner_triggered": ref_triggered, "refiner_correct": ref_correct,
    "strategy_breakdown": strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")


In [ ]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

g_corr = [r["vote_consistency"] for r in all_results  if r["correct"]]
b_corr = [r["vote_consistency"] for r in base_results if r["correct"]]

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (ASDiv)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes per question):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Lift     : {lift:.2f}x  (guided produces {lift:.1f}x more correct votes per question)")
print(f"\n  Per-question comparison (same questions, both modes):")
print(f"    Guided beats baseline : {guided_wins} / {len(all_results)} questions")
print(f"    Baseline beats guided : {baseline_wins} / {len(all_results)} questions")
print(f"    Equal                 : {tied} / {len(all_results)} questions")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gv, bv = g_dist[bkt], b_dist[bkt]
    sign = "+" if gv-bv >= 0 else ""
    print(f"  {bkt:<22} | {gv:>8} | {bv:>8} | {sign+str(gv-bv):>6}")
if g_corr:
    print(f"\n  Correct-question consistency: Guided={np.mean(g_corr)*100:.1f}%  Baseline={np.mean(b_corr)*100:.1f}%")
    print("    (High consistency + correct = genuine reliable solving, not lucky vote)")

angle2 = {
    "dataset": "ASDiv", "n_questions": len(all_results),
    "guided_mean_consistency": round(g_mean,4), "baseline_mean_consistency": round(b_mean,4),
    "consistency_lift": round(lift,4), "guided_wins": guided_wins,
    "baseline_wins": baseline_wins, "tied": tied,
    "guided_distribution": g_dist, "baseline_distribution": b_dist,
    "guided_correct_q_consistency":   round(np.mean(g_corr),4) if g_corr else 0,
    "baseline_correct_q_consistency": round(np.mean(b_corr),4) if b_corr else 0,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")


In [ ]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})
    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"  {'ECE':<26}   {ece:.4f}")
    print(f"  High-conf: {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    return ece, calib_out, false_conf


print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (ASDiv)")
print("=" * 65)
print("Ideal: accuracy at each confidence level matches that level.")
print("False confidence: model agrees on wrong answer with full certainty.")

g_ece, g_calib, g_false = calibration_report(all_results,  "GUIDED pipeline")
b_ece, b_calib, b_false = calibration_report(base_results, "BASELINE (no plan)")

improve = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE Summary:")
print(f"    Guided   ECE : {g_ece:.4f}")
print(f"    Baseline ECE : {b_ece:.4f}")
print(f"    Improvement  : {improve:.1f}% better calibrated")
print(f"\n  False Confidence: Guided={g_false}  Baseline={b_false}  Reduction={b_false-g_false}")

angle3 = {
    "dataset": "ASDiv", "n_questions": len(all_results),
    "guided_ece": round(g_ece,4), "baseline_ece": round(b_ece,4),
    "ece_improvement_pct": round(improve,2),
    "guided_false_confidence": g_false, "baseline_false_confidence": b_false,
    "false_conf_reduction": b_false-g_false,
    "guided_calibration": g_calib, "baseline_calibration": b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")


In [ ]:
# CELL 16 -- ANGLE 4: ACCURACY BY OPERATION TYPE  (ASDiv exclusive)
# =================================================================
# This is the analysis only ASDiv enables.
# We measure guided vs baseline accuracy and vote consistency
# separately for each operation category:
#   Addition / Subtraction / Multiplication / Division / Multi-step
#
# This answers: "Does guidance help more on harder operation types?"
# Expected finding: guidance lifts more on Multiplication and Division
# than on Addition, because those require more structured reasoning.
# =================================================================

g_by_op = defaultdict(list)
b_by_op = defaultdict(list)
for r in all_results:
    g_by_op[r.get("op_type", "Unknown")].append(r)
for r in base_results:
    b_by_op[r.get("op_type", "Unknown")].append(r)

all_ops = sorted(set(list(g_by_op.keys()) + list(b_by_op.keys())))

print("=" * 72)
print("ANGLE 4 -- ACCURACY BY OPERATION TYPE  (ASDiv exclusive)")
print("=" * 72)
print(f"\n  {'Operation':<16} | {'N':>4} | {'Guided':>8} | {'Baseline':>9} | {'Gain':>6} | {'Consistency lift':>17}")
print(f"  {'-'*16}-+-{'-'*4}-+-{'-'*8}-+-{'-'*9}-+-{'-'*6}-+-{'-'*17}")

op_results = {}
for op in all_ops:
    g_items = g_by_op.get(op, [])
    b_items = b_by_op.get(op, [])
    n = len(g_items)
    if n == 0: continue

    g_acc  = sum(r["correct"] for r in g_items) / n * 100
    b_acc  = sum(r["correct"] for r in b_items) / max(len(b_items),1) * 100
    gain   = g_acc - b_acc
    sign   = "+" if gain >= 0 else ""

    g_cons = np.mean([r["vote_consistency"] for r in g_items]) * 100
    b_cons = np.mean([r["vote_consistency"] for r in b_items]) * 100 if b_items else 0
    c_lift = g_cons / max(b_cons, 1e-6)

    print(f"  {op:<16} | {n:>4} | {g_acc:>7.1f}% | {b_acc:>8.1f}% | {sign+str(round(gain,1))+'%':>6} | {c_lift:>6.2f}x  ({g_cons:.1f}% vs {b_cons:.1f}%)")
    op_results[op] = {
        "n": n,
        "guided_acc": round(g_acc,2), "baseline_acc": round(b_acc,2),
        "gain": round(gain,2),
        "guided_consistency": round(g_cons,2), "baseline_consistency": round(b_cons,2),
        "consistency_lift": round(c_lift,4),
    }

# Summary: which operation type benefits most from guidance?
gains = sorted(op_results.items(), key=lambda x: -x[1]["gain"])
print(f"\n  Operation types ranked by guidance gain:")
for op, v in gains:
    sign = "+" if v["gain"] >= 0 else ""
    print(f"    {op:<16}: {sign}{v['gain']}%  (guided={v['guided_acc']}%  baseline={v['baseline_acc']}%)")

angle4 = {"dataset": "ASDiv", "by_operation_type": op_results}
with open(CONFIG["angle4_file"], "w") as f:
    json.dump(angle4, f, indent=2)
print(f"\nSaved -> {CONFIG['angle4_file']}")


In [ ]:
# CELL 17 -- Full Paper Summary (all four angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)
with open(CONFIG["angle4_file"]) as f: a4 = json.load(f)

n = a1["n_questions"]

print("=" * 68)
print("  ASDiv EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset: ASDiv  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print()

rows = [
    ["Metric",                   "Baseline",      "Guided",         "Change"],
    ["Overall Accuracy",
     str(a1['baseline_accuracy']) + "%",
     str(a1['guided_accuracy']) + "%",
     "+" + str(round(a1['guided_accuracy']-a1['baseline_accuracy'],1)) + " pts"],
    ["Compute Cost",
     str(a1['baseline_compute_B']) + "B param-passes",
     str(a1['guided_compute_B']) + "B param-passes",
     str(a1['compute_savings_pct']) + "% cheaper than ceiling"],
    ["Wasted Votes",
     str(a1['baseline_wasted_votes']),
     str(a1['guided_wasted_votes']),
     str(a1['wasted_votes_saved']) + " fewer"],
    ["Vote Consistency",
     str(round(a2['baseline_mean_consistency']*100,1)) + "%",
     str(round(a2['guided_mean_consistency']*100,1)) + "%",
     str(round(a2['consistency_lift'],2)) + "x lift"],
    ["High-Agreement Questions",
     str(a2['baseline_distribution']['high   (80-100%)']),
     str(a2['guided_distribution']['high   (80-100%)']),
     ""],
    ["Guided Wins Per-Question",
     "--",
     str(a2['guided_wins']) + " / " + str(n),
     ""],
    ["ECE (lower = better)",
     str(a3['baseline_ece']),
     str(a3['guided_ece']),
     str(a3['ece_improvement_pct']) + "% better"],
    ["False Confidence Count",
     str(a3['baseline_false_confidence']),
     str(a3['guided_false_confidence']),
     str(a3['false_conf_reduction']) + " fewer"],
]

col_w = [28, 20, 20, 28]
sep   = "-+-".join("-" * w for w in col_w)
for i, row in enumerate(rows):
    line = " | ".join(str(cell).ljust(col_w[j]) for j, cell in enumerate(row))
    print("  " + line)
    if i == 0:
        print("  " + sep)

if a1.get("refiner_triggered", 0) > 0:
    rt = a1["refiner_triggered"]
    rc = a1.get("refiner_correct", 0)
    print(f"\n  Refiner: triggered {rt} times, resolved {rc} correctly ({rc/rt*100:.1f}%)")

# Angle 4 summary
print(f"\n  Accuracy gain by operation type (Angle 4):")
gains = sorted(a4["by_operation_type"].items(), key=lambda x: -x[1]["gain"])
for op, v in gains:
    sign = "+" if v["gain"] >= 0 else ""
    print(f"    {op:<16}: {sign}{v['gain']}%  "
          f"(guided={v['guided_acc']}%  baseline={v['baseline_acc']}%  "
          f"n={v['n']})")

full = {
    "dataset": "ASDiv", "seed": CONFIG["random_seed"],
    "n_questions": n, "angle1": a1, "angle2": a2, "angle3": a3, "angle4": a4,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(full, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}")
print("Commit this notebook to preserve outputs.")


### Part 03

# Notebook 10 -- Three-Angle Evaluation on ARC-Challenge
## SLM-to-SLM Guided Reasoning Pipeline

**Why ARC-Challenge?**

ARC-Challenge (AI2 Reasoning Challenge) is a carefully curated science QA benchmark.
Questions were specifically filtered to EXCLUDE those solvable by simple retrieval or
word-matching — only questions that require genuine multi-hop reasoning remain.

This makes it an ideal stress test for the guided reasoning pipeline outside of
mathematics. If verbal planning helps here, it shows the benefit is NOT math-specific
but rather a general property of structured reasoning.

**Pipeline Under Test:**
- Guide  : Qwen 2.5-3B-Instruct + LoRA fine-tuned adapter (1 forward pass)
- Solver : Qwen 2.5-1.5B-Instruct × 5 majority-vote passes
- Baseline: Qwen 2.5-1.5B-Instruct × 5 majority-vote passes (no guide plan)
- Random chance: 25.0% (4 options: A, B, C, D)

**Three Evaluation Angles:**
1. Compute Efficiency  -- accuracy per billion parameter-passes
2. Vote Consistency    -- how reliably the ensemble agrees on the correct answer
3. Confidence Calibration -- how well confidence predicts correctness (ECE)


In [ ]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "allenai/ai2_arc",
    "dataset_config"      : "ARC-Challenge",   # NOT ARC-Easy
    "dataset_split"       : "test",            # 1172 test questions
    "max_eval_samples"    : 900,
    "random_seed"         : 42,                # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 400,

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


In [ ]:
# CELL 5 -- Load ARC-Challenge dataset
# ARC fields:
#   question   : str  (the question text)
#   choices    : dict with keys 'text' (list of str) and 'label' (list of str like ['A','B','C','D'])
#   answerKey  : str  (single letter, always uppercase -- can be '1','2','3','4' in rare cases)
#
# Some records use numeric labels (1,2,3,4) instead of letters -- we normalise all to A,B,C,D.
# We format: "<question>\n\nOptions:\nA) ...\nB) ...\nC) ...\nD) ..."

print("Loading ARC-Challenge from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_config"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Features : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Test size: {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record:")
for k, v in ex.items():
    print(f"  {k}: {v}")


# Numeric label -> letter mapping (rare but present in ARC)
NUM_TO_LETTER = {"1": "A", "2": "B", "3": "C", "4": "D", "5": "E"}

def normalise_label(label):
    """Convert '1','2','3','4' to 'A','B','C','D' if needed."""
    s = str(label).strip().upper()
    return NUM_TO_LETTER.get(s, s)

def normalise_arc(item):
    """Convert ARC record to {question, answer} used by the pipeline."""
    labels = [normalise_label(l) for l in item["choices"]["label"]]
    texts  = item["choices"]["text"]
    options_str = "\n".join(f"{l}) {t}" for l, t in zip(labels, texts))
    q = item["question"].strip() + "\n\nOptions:\n" + options_str
    ans = normalise_label(item["answerKey"])
    return {"question": q, "answer": ans, "raw_choices": list(zip(labels, texts))}


all_data = [normalise_arc(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Determine valid letters from actual data (usually A-D, occasionally A-E)
all_labels = set()
for item in all_data:
    for lbl, _ in item["raw_choices"]:
        all_labels.add(lbl)
VALID_LETTERS = all_labels
print(f"\nValid answer letters found: {sorted(VALID_LETTERS)}")

# CRITICAL: fix seed ONCE before sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"Sampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"Using all {len(test_data)} questions")

print(f"\nSample question:\n{test_data[0]['question']}")
print(f"Answer: {test_data[0]['answer']}")


In [ ]:
# CELL 6 -- Answer extraction for multiple-choice (A-D)
# ARC-Challenge answers are single letters A, B, C, or D.

def extract_gt_answer(answer_str):
    """GT is already a clean letter -- just uppercase and validate."""
    s = normalise_label(str(answer_str).strip())
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    """
    Extract the chosen option letter (A-D) from model free-form output.
    Priority order -- most explicit formats first.
    """
    text = text.strip()

    # 1. Conclusive answer phrases
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is)"
        r"[\s:]*([A-D])\b",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 2. "option/choice X is correct/is the answer"
    m = re.search(
        r"(?:option|choice)\s+([A-D])\s+(?:is correct|is the answer|matches|is right)",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 3. #### A  -- standard termination marker
    m = re.search(r"####\s*([A-D])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 4. Parenthesised at end of text: (A), (B) ...
    m = re.search(r"\(([A-D])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 5. Bold: **A**, **A)**
    m = re.search(r"\*\*([A-D])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 6. Standalone letter on its own line (last occurrence)
    matches = re.findall(r"^\s*([A-D])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    # 7. Last standalone letter anywhere
    matches = re.findall(r"\b([A-D])\b", text, re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    return ""

# Quick test
test_outputs = [
    "After reasoning through the options, the answer is C",
    "The correct answer is B.",
    "#### D",
    "(A)",
    "**B**",
]
for t in test_outputs:
    print(f"  '{t[:50]}' -> '{extract_pred_answer(t)}'")
print("Extraction OK")


In [ ]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        ADAPTER_PATH,
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token







if 'guide_model' in dir():
    del guide_model
if 'adapter_path' in dir():
    del adapter_path
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Loading guide base: {CONFIG['guide_base']}")




guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


In [ ]:
# CELL 9 -- Prompts and generation functions
# ARC-Challenge specific: guide breaks down the SCIENCE REASONING steps needed.
# Unlike math, there is no numeric target -- instead the guide identifies:
#   1. The key scientific concept being tested
#   2. Relevant facts or principles that apply
#   3. Which options are likely correct / eliminable

GUIDE_SYSTEM = (
    "You are a science reasoning assistant for multiple-choice questions.\n"
    "Given a science question with options A-D, write 2-3 concrete reasoning steps.\n"
    "Each step must identify SPECIFIC scientific facts, principles, or definitions.\n"
    "Your LAST line must always be: Best answer: <letter> because <one-line reason>\n\n"
    "Rules:\n"
    "- Steps must reference exact concepts from the question and options.\n"
    "- No vague steps like 'think about energy'. Be specific.\n"
    "- Eliminate wrong options explicitly when possible.\n"
    "- No LaTeX. No markdown. Plain text only.\n\n"
    "BAD example (too vague):\n"
    "  Step 1: Think about what the question is asking.\n"
    "  Step 2: Consider the properties of matter.\n"
    "  Best answer: C because it seems right\n\n"
    "GOOD example (specific reasoning):\n"
    "  Step 1: The question asks what happens to molecules when water freezes.\n"
    "           Freezing = liquid to solid = molecules slow down and form fixed lattice.\n"
    "  Step 2: Option A says molecules speed up -- wrong, freezing slows them.\n"
    "           Option C says they arrange in a regular pattern -- matches crystalline solid.\n"
    "  Best answer: C because water molecules form an ordered lattice structure when frozen\n\n"
    "Apply this pattern to any science question -- biology, chemistry, physics, earth science."
)

SOLVE_SYSTEM = (
    "You are a precise multiple-choice science solver.\n"
    "You are given a science question and a reasoning plan.\n"
    "Follow the plan steps exactly and pick the letter the plan identifies as correct.\n"
    "Do not contradict the plan. Do not re-examine eliminated options.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]\n\n"
    "Example:\n"
    "Plan says: Best answer: C because molecules form an ordered lattice when frozen.\n"
    "The answer is C"
)

SOLVE_BASELINE_SYSTEM = (
    "You are a science question answering assistant.\n"
    "Read the question carefully. Use your knowledge to pick the best answer.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful science reasoning checker.\n"
    "You are given a science question and a list of candidate answers that are tied.\n"
    "Reason step by step about which answer is most scientifically accurate.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)


def _generate(model, tokenizer, system_prompt, user_prompt, temperature, max_new_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def generate_plan(question):
    return _generate(
        guide_model, guide_tok,
        GUIDE_SYSTEM, question,
        CONFIG["guide_temperature"], CONFIG["max_new_tokens"]
    )

def generate_guided(question, plan):
    prompt = f"Plan:\n{plan}\n\nNow answer:\n{question}"
    return _generate(
        resp_model, resp_tok,
        SOLVE_SYSTEM, prompt,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_baseline(question):
    return _generate(
        resp_model, resp_tok,
        SOLVE_BASELINE_SYSTEM, question,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_refiner(question, tied_answers):
    prompt = (
        f"Question:\n{question}\n\n"
        f"Tied candidate answers: {', '.join(tied_answers)}\n"
        f"Which one is most scientifically accurate?"
    )
    return _generate(
        resp_model, resp_tok,
        REFINER_SYSTEM, prompt,
        CONFIG["refiner_temperature"], CONFIG["max_new_tokens"]
    )

print("Prompt and generation functions ready")


In [ ]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Filters empty/invalid letter responses before counting.

    Returns dict with all metrics needed for the three angles.
    """
    valid = [a for a in answers if a in VALID_LETTERS]
    if not valid:
        valid = answers  # fallback

    vote_counts  = Counter(valid)
    most_common  = vote_counts.most_common()
    top_answer   = most_common[0][0]
    top_count    = most_common[0][1]
    total        = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        # Tie: refiner breaks it
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans in VALID_LETTERS else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted     = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "total_votes"      : total,
        "correct_votes"    : correct_votes,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }

print("Voting logic ready")


In [ ]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (ARC-Challenge)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question:\n{q}")
print(f"\nGT Answer: {gt}")

# Guided
print("\n\n\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n\n\n\n\n\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")
print(f"  Vote counts: {g['vote_counts']}")

# Baseline
print("\n\n\n\n\n\n\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print(f"  Vote counts: {b['vote_counts']}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


In [ ]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided   (guide plan + solver x5)
#   Mode B: Baseline (solver x5, no plan)
#
# Checkpointing every save_every questions -- safe to interrupt and resume.

print(f"Dual evaluation: {len(test_data)} ARC-Challenge questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print(f"Random baseline (chance): 25.0% (1 in 4 options)")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="ARC-Challenge Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "plan"             : plan,
            "votes"            : g_votes_raw,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "vote_counts"      : g_dec["vote_counts"],
            "total_votes"      : g_dec["total_votes"],
            "correct_votes"    : g_dec["correct_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [GUIDED ERROR idx={idx}]: {e}")
        all_results.append({"mode":"guided","idx":idx,"correct":False,
                             "gt_answer":gt_answer,"final_answer":"",
                             "strategy":"error","confidence":0.0,
                             "vote_consistency":0.0,"wasted_votes":5,
                             "refiner_used":False,"refiner_correct":None,
                             "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "votes"            : b_votes_raw,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "vote_counts"      : b_dec["vote_counts"],
            "total_votes"      : b_dec["total_votes"],
            "correct_votes"    : b_dec["correct_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : b_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [BASELINE ERROR idx={idx}]: {e}")
        base_results.append({"mode":"baseline","idx":idx,"correct":False,
                              "gt_answer":gt_answer,"final_answer":"",
                              "strategy":"error","confidence":0.0,
                              "vote_consistency":0.0,"wasted_votes":5,
                              "refiner_used":False,"refiner_correct":None,
                              "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- Checkpoint -------------------------------------------
    if (idx + 1) % CONFIG["save_every"] == 0 or (idx + 1) == len(test_data):
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        elapsed = time.time() - t0
        g_acc_so_far = sum(r["correct"] for r in all_results) / len(all_results) * 100
        b_acc_so_far = sum(r["correct"] for r in base_results) / len(base_results) * 100
        print(f"  [{idx+1}/{len(test_data)}] Guided: {g_acc_so_far:.1f}%  "
              f"Baseline: {b_acc_so_far:.1f}%  ({elapsed/60:.1f}min)")

print("\n" + "=" * 65)
print("EVALUATION COMPLETE")
g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
print(f"  Guided   accuracy: {g_acc:.1f}%")
print(f"  Baseline accuracy: {b_acc:.1f}%")
print(f"  Delta            : +{g_acc - b_acc:.1f} pts")
print(f"  Random chance    : 25.0%")
print("=" * 65)


In [ ]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
# =================================================================
# We compare three setups by accuracy and compute cost:
#   Baseline : 1.5B solver x5 votes              = 7.5B param-passes
#   Guided   : 3B guide x1 + 1.5B solver x5      = 10.5B param-passes
#   Upper    : 3B guide x5 votes (ceiling)        = 15.0B param-passes
#
# ARC-Challenge random chance baseline: 25.0% (4 options A-D)
# =================================================================

G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]

guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N
random_chance    = 25.0   # 1 in 4 options

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

g_eff = g_acc / guided_compute
b_eff = b_acc / baseline_compute
savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted = sum(r["wasted_votes"] for r in all_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)
total_possible = len(all_results) * N

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (ARC-Challenge)")
print("=" * 65)
print(f"  Random chance baseline: {random_chance}% (4 options, A-D)")
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}")
print(f"  {'Random chance':<32} | {'--':>10} | {random_chance:>8.1f}%")
print(f"  {'Baseline (1.5B x ' + str(N) + ')':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}%")
print(f"  {'Guided  (3B x1 + 1.5B x' + str(N) + ')':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}%")
print(f"  {'Upper   (3B x ' + str(N) + ')':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9}")
print(f"\n  Guided vs baseline gain : +{g_acc - b_acc:.1f} pts")
print(f"  Guided vs random chance : +{g_acc - random_chance:.1f} pts above chance")
print(f"  Compute savings vs upper: {savings_pct:.0f}% cheaper")
print(f"  Wasted votes saved      : {b_wasted - g_wasted}")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")

print("\n  Strategy breakdown:")
for strat, stats in strategy_stats.items():
    acc = stats["correct"] / stats["n"] * 100 if stats["n"] else 0
    print(f"    {strat:<20}: {stats['n']}q  |  {acc:.1f}% accurate")

a1_data = {
    "dataset": "ARC-Challenge",
    "n_questions": len(all_results),
    "random_chance": random_chance,
    "guided_accuracy": round(g_acc, 1),
    "baseline_accuracy": round(b_acc, 1),
    "accuracy_gain": round(g_acc - b_acc, 1),
    "guided_above_chance": round(g_acc - random_chance, 1),
    "baseline_above_chance": round(b_acc - random_chance, 1),
    "guided_compute_B": guided_compute,
    "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute,
    "compute_savings_pct": round(savings_pct, 1),
    "guided_wasted_votes": g_wasted,
    "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted - g_wasted,
    "refiner_triggered": ref_triggered,
    "refiner_correct": ref_correct,
    "strategy_stats": strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(a1_data, f, indent=2)
print(f"\nAngle 1 saved to {CONFIG['angle1_file']}")


In [ ]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
# =================================================================
# Vote consistency = fraction of votes (out of 5) that matched GT.
# High consistency = solver reliably produces correct answer.
# Low consistency  = got lucky with majority vote.
# =================================================================

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

# Letter distribution (check for option position bias)
all_letters_guided   = []
all_letters_baseline = []
for r in all_results:
    all_letters_guided.extend(r["vote_counts"].keys())
for r in base_results:
    all_letters_baseline.extend(r["vote_counts"].keys())
g_letter_dist = Counter(all_letters_guided)
b_letter_dist = Counter(all_letters_baseline)

# Normalize letter distribution to percentages
total_g_letters = sum(g_letter_dist.values())
total_b_letters = sum(b_letter_dist.values())

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (ARC-Challenge)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Lift     : {lift:.2f}x")
print(f"\n  Per-question: Guided wins {guided_wins}, Baseline wins {baseline_wins}, Tied {tied}")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gn, bn = g_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {gn:>8} | {bn:>8} | {gn-bn:>+6}")

print(f"\n  Option letter distribution (% of winning votes):")
print(f"  {'Letter':<8} | {'Guided':>8} | {'Baseline':>8}")
print(f"  {'-'*8}-+-{'-'*8}-+-{'-'*8}")
for ltr in sorted(VALID_LETTERS):
    gp = g_letter_dist.get(ltr, 0) / max(total_g_letters, 1) * 100
    bp = b_letter_dist.get(ltr, 0) / max(total_b_letters, 1) * 100
    print(f"  {ltr:<8} | {gp:>7.1f}% | {bp:>7.1f}%")

# Complete failure (all 5 votes wrong)
all_wrong_guided   = sum(1 for r in all_results  if r["vote_consistency"] == 0.0)
all_wrong_baseline = sum(1 for r in base_results if r["vote_consistency"] == 0.0)
print(f"\n  All-wrong questions (0 of 5 votes correct):")
print(f"    Guided  : {all_wrong_guided}  |  Baseline: {all_wrong_baseline}  |  Delta: {all_wrong_baseline - all_wrong_guided} fewer failures with guidance")

a2_data = {
    "dataset": "ARC-Challenge",
    "guided_mean_consistency": round(g_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "guided_wins": guided_wins,
    "baseline_wins": baseline_wins,
    "tied": tied,
    "guided_distribution": g_dist,
    "baseline_distribution": b_dist,
    "all_wrong_guided": all_wrong_guided,
    "all_wrong_baseline": all_wrong_baseline,
    "guided_letter_dist": {k: round(v/max(total_g_letters,1)*100,1) for k,v in g_letter_dist.items()},
    "baseline_letter_dist": {k: round(v/max(total_b_letters,1)*100,1) for k,v in b_letter_dist.items()},
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(a2_data, f, indent=2)
print(f"\nAngle 2 saved to {CONFIG['angle2_file']}")


In [ ]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
# =================================================================
# Confidence = fraction of votes that agreed on the winning answer.
# Perfect calibration: 80% confidence -> 80% accuracy.
#
# ECE (Expected Calibration Error): average |accuracy - confidence|
# weighted by bucket size. Lower = better calibrated.
#
# ARC-Challenge note: with 4 options, random confidence is 25%.
# False confidence (all 5 agree but wrong) is especially damaging --
# it gives no signal to distrust the answer for downstream use.
# =================================================================

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})

    hc = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"  {'ECE':<26}   {ece:.4f}")
    print(f"  High-conf: {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    return ece, calib_out, false_conf, hc_acc, len(hc)

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (ARC-Challenge)")
print("=" * 65)
g_ece, g_calib, g_fc, g_hc_acc, g_hc_n = calibration_report(all_results, "GUIDED")
b_ece, b_calib, b_fc, b_hc_acc, b_hc_n = calibration_report(base_results, "BASELINE")

ece_improvement = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE improvement : {ece_improvement:.1f}% better calibrated with guidance")
print(f"  ECE threshold   : Guided {'PASSES' if g_ece < 0.10 else 'FAILS'} the <0.10 threshold")
print(f"  False confidence: Guided {g_fc}  vs  Baseline {b_fc}  ({b_fc - g_fc} fewer with guidance)")

a3_data = {
    "dataset": "ARC-Challenge",
    "guided_ece": round(g_ece, 4),
    "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(ece_improvement, 1),
    "guided_calibration": g_calib,
    "baseline_calibration": b_calib,
    "guided_false_confidence": g_fc,
    "baseline_false_confidence": b_fc,
    "guided_high_conf_accuracy": round(g_hc_acc, 1),
    "baseline_high_conf_accuracy": round(b_hc_acc, 1),
    "guided_high_conf_n": g_hc_n,
    "baseline_high_conf_n": b_hc_n,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(a3_data, f, indent=2)
print(f"\nAngle 3 saved to {CONFIG['angle3_file']}")


In [ ]:
# CELL 16 -- Full Paper Summary (all three angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n = a1["n_questions"]

print("=" * 68)
print("  ARC-CHALLENGE EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset: ARC-Challenge  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print(f"  Random chance baseline: 25.0%  (4 options, A-D)")
print()

rows = [
    ["Metric",                   "Baseline",                  "Guided",                     "Change"],
    ["Overall Accuracy",
     str(a1["baseline_accuracy"]) + "%",
     str(a1["guided_accuracy"]) + "%",
     "+" + str(round(a1["guided_accuracy"]-a1["baseline_accuracy"],1)) + " pts"],
    ["Above Random Chance (25%)",
     "+" + str(a1["baseline_above_chance"]) + " pts",
     "+" + str(a1["guided_above_chance"]) + " pts",
     ""],
    ["Compute Cost",
     str(a1["baseline_compute_B"]) + "B param-passes",
     str(a1["guided_compute_B"]) + "B param-passes",
     str(a1["compute_savings_pct"]) + "% cheaper than ceiling"],
    ["Wasted Votes",
     str(a1["baseline_wasted_votes"]),
     str(a1["guided_wasted_votes"]),
     str(a1["wasted_votes_saved"]) + " fewer"],
    ["Vote Consistency",
     str(round(a2["baseline_mean_consistency"]*100,1)) + "%",
     str(round(a2["guided_mean_consistency"]*100,1)) + "%",
     str(round(a2["consistency_lift"],2)) + "x lift"],
    ["High-Agreement Questions",
     str(a2["baseline_distribution"]["high   (80-100%)"]),
     str(a2["guided_distribution"]["high   (80-100%)"]),
     ""],
    ["All-Wrong (0/5 votes)",
     str(a2["all_wrong_baseline"]),
     str(a2["all_wrong_guided"]),
     str(a2["all_wrong_baseline"] - a2["all_wrong_guided"]) + " fewer complete failures"],
    ["ECE (lower = better)",
     str(a3["baseline_ece"]),
     str(a3["guided_ece"]),
     str(a3["ece_improvement_pct"]) + "% better"],
    ["High-Conf Accuracy (>=0.80)",
     str(a3["baseline_high_conf_accuracy"]) + "% (n=" + str(a3["baseline_high_conf_n"]) + ")",
     str(a3["guided_high_conf_accuracy"])  + "% (n=" + str(a3["guided_high_conf_n"])  + ")",
     ""],
    ["False Confidence Count",
     str(a3["baseline_false_confidence"]),
     str(a3["guided_false_confidence"]),
     str(a3["baseline_false_confidence"] - a3["guided_false_confidence"]) + " fewer"],
]

col_w = [32, 26, 26, 30]
header = rows[0]
sep    = "  " + "-+-".join("-" * w for w in col_w)
print("  " + " | ".join(f"{h:<{col_w[i]}}" for i, h in enumerate(header)))
print(sep)
for row in rows[1:]:
    print("  " + " | ".join(f"{str(row[i]):<{col_w[i]}}" for i in range(len(col_w))))

print()
print("=" * 68)
print("  KEY FINDING:")
print(f"  The guided pipeline achieves +{a1['accuracy_gain']:.1f} pts accuracy improvement")
print(f"  on ARC-Challenge (science reasoning) at only {a1['guided_compute_B']}B param-passes,")
print(f"  which is {a1['compute_savings_pct']:.0f}% cheaper than the {a1['upper_compute_B']}B upper bound.")
print(f"  Calibration improves by {a3['ece_improvement_pct']:.1f}% (ECE: {a3['baseline_ece']} -> {a3['guided_ece']}).")
print(f"  This result holds outside mathematics, demonstrating that structured verbal")
print(f"  planning generalizes across reasoning domains.")
print("=" * 68)


### Part 04

# Notebook 11 -- Three-Angle Evaluation on CommonsenseQA
## SLM-to-SLM Guided Reasoning Pipeline

**Why CommonsenseQA?**

CommonsenseQA tests everyday real-world reasoning — the kind of knowledge
humans pick up from lived experience rather than formal education.
Questions require understanding concepts like causality, spatial relationships,
social norms, and object properties.

This is maximally different from math: there are no equations, no numeric
targets, no formal operations. If the guided pipeline helps here, it
strongly suggests the benefit comes from **structured verbal reasoning**
in general — not from domain-specific fine-tuning knowledge.

The guide was fine-tuned on GSM8K (math). CommonsenseQA is the
hardest possible out-of-domain test for that guide.

**Pipeline Under Test:**
- Guide  : Qwen 2.5-3B-Instruct + LoRA fine-tuned adapter (1 forward pass)
- Solver : Qwen 2.5-1.5B-Instruct × 5 majority-vote passes
- Baseline: Qwen 2.5-1.5B-Instruct × 5 majority-vote passes (no guide plan)
- Random chance: **20.0%** (5 options: A–E)

**Note on dataset split:**
The CommonsenseQA official test labels are withheld for the public leaderboard.
We use the **validation split** (1,221 questions with ground-truth labels),
which is standard practice for offline evaluation.

**Three Evaluation Angles:**
1. Compute Efficiency  -- accuracy per billion parameter-passes
2. Vote Consistency    -- how reliably the ensemble agrees on the correct answer
3. Confidence Calibration -- how well confidence predicts correctness (ECE)


In [ ]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "tau/commonsense_qa",
    "dataset_split"       : "validation",  # test labels are withheld; validation has 1,221 questions
    "max_eval_samples"    : 900,
    "random_seed"         : 42,            # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 400,

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


In [ ]:
# CELL 5 -- Load CommonsenseQA dataset
# Fields:
#   id          : str
#   question    : str  (the question text)
#   question_concept : str (the core concept being tested)
#   choices     : dict with 'label' (list: ['A','B','C','D','E']) and 'text' (list of str)
#   answerKey   : str  (single letter A-E, only available in train/validation)
#
# We use the validation split (1,221 questions with labels).
# Format: "<question>\n\nOptions:\nA) ...\nB) ...\nC) ...\nD) ...\nE) ..."

print("Loading CommonsenseQA from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Features : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Val size : {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record:")
for k, v in ex.items():
    print(f"  {k}: {v}")


VALID_LETTERS = set("ABCDE")

def normalise_csqa(item):
    """Convert CommonsenseQA record to {question, answer, concept} for the pipeline."""
    labels = item["choices"]["label"]   # ['A','B','C','D','E']
    texts  = item["choices"]["text"]    # list of option strings
    options_str = "\n".join(f"{l}) {t}" for l, t in zip(labels, texts))
    q   = item["question"].strip() + "\n\nOptions:\n" + options_str
    ans = str(item["answerKey"]).strip().upper()
    return {
        "question"  : q,
        "answer"    : ans,
        "concept"   : item.get("question_concept", ""),
    }


all_data = [normalise_csqa(x) for x in raw_ds[CONFIG["dataset_split"]]]

# CRITICAL: fix seed ONCE before sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

print(f"\nSample question:\n{test_data[0]['question']}")
print(f"Concept : {test_data[0]['concept']}")
print(f"Answer  : {test_data[0]['answer']}")


In [ ]:
# CELL 6 -- Answer extraction for multiple-choice (A-E)
# CommonsenseQA answers are single letters A, B, C, D, or E.
# Extraction logic mirrors AQUA-RAT (also 5-choice).

def extract_gt_answer(answer_str):
    """GT is already a clean letter -- just uppercase and validate."""
    s = str(answer_str).strip().upper()
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    """
    Extract the chosen option letter (A-E) from model free-form output.
    Priority order -- most explicit formats first.
    """
    text = text.strip()

    # 1. Conclusive answer phrases
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is)"
        r"[\s:]*([A-E])\b",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 2. "option/choice X is correct/is the answer"
    m = re.search(
        r"(?:option|choice)\s+([A-E])\s+(?:is correct|is the answer|matches|is right|is most likely)",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 3. #### A  -- standard termination marker
    m = re.search(r"####\s*([A-E])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 4. Parenthesised at end: (A), (B) ...
    m = re.search(r"\(([A-E])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 5. Bold: **A**, **A)**
    m = re.search(r"\*\*([A-E])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 6. Standalone letter on its own line (last occurrence)
    matches = re.findall(r"^\s*([A-E])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    # 7. Last standalone letter anywhere
    matches = re.findall(r"\b([A-E])\b", text, re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    return ""

# Quick test
test_outputs = [
    "After thinking it through, the answer is C",
    "The correct answer is B.",
    "#### D",
    "(A)",
    "**E**",
]
for t in test_outputs:
    print(f"  '{t[:50]}' -> '{extract_pred_answer(t)}'")
print("Extraction OK")


In [ ]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        ADAPTER_PATH,
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token






if 'guide_model' in dir():
    del guide_model
if 'adapter_path' in dir():
    del adapter_path
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Loading guide base: {CONFIG['guide_base']}")




guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


In [ ]:
# CELL 9 -- Prompts and generation functions
# CommonsenseQA specific: guide reasons about everyday concepts, not math.
# The guide identifies:
#   1. The core concept being tested
#   2. Relevant real-world knowledge that applies
#   3. Which options match or contradict that knowledge

GUIDE_SYSTEM = (
    "You are a commonsense reasoning assistant for multiple-choice questions.\n"
    "Given a question with options A-E, write 2-3 concrete reasoning steps.\n"
    "Each step must use specific real-world knowledge relevant to the question.\n"
    "Your LAST line must always be: Best answer: <letter> because <one-line reason>\n\n"
    "Rules:\n"
    "- Steps must reference SPECIFIC facts, not vague generalities.\n"
    "- Explicitly eliminate at least one wrong option when possible.\n"
    "- Think about cause-effect, typical human behavior, object properties, or spatial logic.\n"
    "- No markdown. Plain text only.\n\n"
    "BAD example (too vague):\n"
    "  Step 1: Think about the concept in the question.\n"
    "  Step 2: Consider which option makes sense.\n"
    "  Best answer: C because it seems right\n\n"
    "GOOD example (specific reasoning):\n"
    "  Question: Where would you find a penguin in its natural habitat?\n"
    "  Options: A) Amazon rainforest B) Antarctic ice C) Sahara desert D) Rocky mountains E) Coral reef\n"
    "  Step 1: Penguins are native to the Southern Hemisphere and thrive in cold climates.\n"
    "           They live on ice and hunt fish in cold ocean waters.\n"
    "  Step 2: Options A, C, D, E are all warm or tropical -- penguins cannot survive there.\n"
    "           Only option B matches: cold, icy, near ocean.\n"
    "  Best answer: B because penguins are cold-climate birds native to Antarctica\n\n"
    "Apply this pattern to any commonsense question -- cause-effect, location, social norms, "
    "object use, human behavior, or physical properties."
)

SOLVE_SYSTEM = (
    "You are a precise multiple-choice commonsense solver.\n"
    "You are given a question and a reasoning plan.\n"
    "Follow the plan exactly and pick the letter the plan identifies as correct.\n"
    "Do not contradict the plan. Do not reconsider eliminated options.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]\n\n"
    "Example:\n"
    "Plan says: Best answer: B because penguins are cold-climate birds native to Antarctica.\n"
    "The answer is B"
)

SOLVE_BASELINE_SYSTEM = (
    "You are a commonsense question answering assistant.\n"
    "Read the question carefully. Use your general knowledge to pick the best answer.\n"
    "Think step by step if needed.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful commonsense reasoning checker.\n"
    "You are given a question and a list of candidate answers that are tied.\n"
    "Reason step by step about which answer is most plausible given real-world knowledge.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)


def _generate(model, tokenizer, system_prompt, user_prompt, temperature, max_new_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def generate_plan(question):
    return _generate(
        guide_model, guide_tok,
        GUIDE_SYSTEM, question,
        CONFIG["guide_temperature"], CONFIG["max_new_tokens"]
    )

def generate_guided(question, plan):
    prompt = f"Plan:\n{plan}\n\nNow answer:\n{question}"
    return _generate(
        resp_model, resp_tok,
        SOLVE_SYSTEM, prompt,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_baseline(question):
    return _generate(
        resp_model, resp_tok,
        SOLVE_BASELINE_SYSTEM, question,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_refiner(question, tied_answers):
    prompt = (
        f"Question:\n{question}\n\n"
        f"Tied candidate answers: {', '.join(tied_answers)}\n"
        f"Which one is most plausible based on common sense?"
    )
    return _generate(
        resp_model, resp_tok,
        REFINER_SYSTEM, prompt,
        CONFIG["refiner_temperature"], CONFIG["max_new_tokens"]
    )

print("Prompts and generation functions ready")


In [ ]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Filters empty/invalid letter responses before counting.
    Returns dict with all metrics needed for the three angles.
    """
    valid = [a for a in answers if a in VALID_LETTERS]
    if not valid:
        valid = answers  # fallback

    vote_counts  = Counter(valid)
    most_common  = vote_counts.most_common()
    top_answer   = most_common[0][0]
    top_count    = most_common[0][1]
    total        = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans in VALID_LETTERS else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted     = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "total_votes"      : total,
        "correct_votes"    : correct_votes,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }

print("Voting logic ready")


In [ ]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (CommonsenseQA)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question:\n{q}")
print(f"Concept : {item['concept']}")
print(f"GT Answer: {gt}")

# Guided
print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")
print(f"  Vote counts: {g['vote_counts']}")

# Baseline
print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print(f"  Vote counts: {b['vote_counts']}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


In [ ]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided   (guide plan + solver x5)
#   Mode B: Baseline (solver x5, no plan)
#
# Checkpointing every save_every questions -- safe to interrupt and resume.

print(f"Dual evaluation: {len(test_data)} CommonsenseQA questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print(f"Random baseline (chance): 20.0% (1 in 5 options)")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="CommonsenseQA Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "concept"          : item.get("concept", ""),
            "gt_answer"        : gt_answer,
            "plan"             : plan,
            "votes"            : g_votes_raw,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "vote_counts"      : g_dec["vote_counts"],
            "total_votes"      : g_dec["total_votes"],
            "correct_votes"    : g_dec["correct_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [GUIDED ERROR idx={idx}]: {e}")
        all_results.append({"mode":"guided","idx":idx,"correct":False,
                             "gt_answer":gt_answer,"final_answer":"",
                             "concept": item.get("concept",""),
                             "strategy":"error","confidence":0.0,
                             "vote_consistency":0.0,"wasted_votes":5,
                             "refiner_used":False,"refiner_correct":None,
                             "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "votes"            : b_votes_raw,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "vote_counts"      : b_dec["vote_counts"],
            "total_votes"      : b_dec["total_votes"],
            "correct_votes"    : b_dec["correct_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : b_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [BASELINE ERROR idx={idx}]: {e}")
        base_results.append({"mode":"baseline","idx":idx,"correct":False,
                              "gt_answer":gt_answer,"final_answer":"",
                              "strategy":"error","confidence":0.0,
                              "vote_consistency":0.0,"wasted_votes":5,
                              "refiner_used":False,"refiner_correct":None,
                              "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- Checkpoint -------------------------------------------
    if (idx + 1) % CONFIG["save_every"] == 0 or (idx + 1) == len(test_data):
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        elapsed = time.time() - t0
        g_acc_so_far = sum(r["correct"] for r in all_results) / len(all_results) * 100
        b_acc_so_far = sum(r["correct"] for r in base_results) / len(base_results) * 100
        print(f"  [{idx+1}/{len(test_data)}] Guided: {g_acc_so_far:.1f}%  "
              f"Baseline: {b_acc_so_far:.1f}%  ({elapsed/60:.1f}min)")

print("\n" + "=" * 65)
print("EVALUATION COMPLETE")
g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
print(f"  Guided   accuracy: {g_acc:.1f}%")
print(f"  Baseline accuracy: {b_acc:.1f}%")
print(f"  Delta            : +{g_acc - b_acc:.1f} pts")
print(f"  Random chance    : 20.0%")
print("=" * 65)
 

In [ ]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
# =================================================================
# Baseline : 1.5B solver x5 votes              = 7.5B param-passes
# Guided   : 3B guide x1 + 1.5B solver x5      = 10.5B param-passes
# Upper    : 3B guide x5 votes (ceiling)        = 15.0B param-passes
# CommonsenseQA random chance: 20.0% (5 options A-E)
# =================================================================

G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]

guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N
random_chance    = 20.0

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted = sum(r["wasted_votes"] for r in all_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (CommonsenseQA)")
print("=" * 65)
print(f"  Random chance baseline: {random_chance}% (5 options, A-E)")
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}")
print(f"  {'Random chance':<32} | {'--':>10} | {random_chance:>8.1f}%")
print(f"  {'Baseline (1.5B x ' + str(N) + ')':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}%")
print(f"  {'Guided  (3B x1 + 1.5B x' + str(N) + ')':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}%")
print(f"  {'Upper   (3B x ' + str(N) + ')':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9}")
print(f"\n  Guided vs baseline gain : +{g_acc - b_acc:.1f} pts")
print(f"  Guided vs random chance : +{g_acc - random_chance:.1f} pts above chance")
print(f"  Compute savings vs upper: {savings_pct:.0f}% cheaper")
print(f"  Wasted votes saved      : {b_wasted - g_wasted}")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")

print("\n  Strategy breakdown:")
for strat, stats in strategy_stats.items():
    acc = stats["correct"] / stats["n"] * 100 if stats["n"] else 0
    print(f"    {strat:<20}: {stats['n']}q  |  {acc:.1f}% accurate")

a1_data = {
    "dataset": "CommonsenseQA",
    "n_questions": len(all_results),
    "random_chance": random_chance,
    "guided_accuracy": round(g_acc, 1),
    "baseline_accuracy": round(b_acc, 1),
    "accuracy_gain": round(g_acc - b_acc, 1),
    "guided_above_chance": round(g_acc - random_chance, 1),
    "baseline_above_chance": round(b_acc - random_chance, 1),
    "guided_compute_B": guided_compute,
    "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute,
    "compute_savings_pct": round(savings_pct, 1),
    "guided_wasted_votes": g_wasted,
    "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted - g_wasted,
    "refiner_triggered": ref_triggered,
    "refiner_correct": ref_correct,
    "strategy_stats": strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(a1_data, f, indent=2)
print(f"\nAngle 1 saved to {CONFIG['angle1_file']}")


In [ ]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
# =================================================================
# Vote consistency = fraction of votes (out of 5) that matched GT.
# Also tracks letter distribution to detect position bias (A-E).
# =================================================================

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

all_letters_guided   = []
all_letters_baseline = []
for r in all_results:
    all_letters_guided.extend(r["vote_counts"].keys())
for r in base_results:
    all_letters_baseline.extend(r["vote_counts"].keys())
g_letter_dist = Counter(all_letters_guided)
b_letter_dist = Counter(all_letters_baseline)

total_g_letters = sum(g_letter_dist.values())
total_b_letters = sum(b_letter_dist.values())

all_wrong_guided   = sum(1 for r in all_results  if r["vote_consistency"] == 0.0)
all_wrong_baseline = sum(1 for r in base_results if r["vote_consistency"] == 0.0)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (CommonsenseQA)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Lift     : {lift:.2f}x")
print(f"\n  Per-question: Guided wins {guided_wins}, Baseline wins {baseline_wins}, Tied {tied}")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gn, bn = g_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {gn:>8} | {bn:>8} | {gn-bn:>+6}")

print(f"\n  Option letter distribution (% of winning votes):")
print(f"  {'Letter':<8} | {'Guided':>8} | {'Baseline':>8}")
print(f"  {'-'*8}-+-{'-'*8}-+-{'-'*8}")
for ltr in sorted(VALID_LETTERS):
    gp = g_letter_dist.get(ltr, 0) / max(total_g_letters, 1) * 100
    bp = b_letter_dist.get(ltr, 0) / max(total_b_letters, 1) * 100
    print(f"  {ltr:<8} | {gp:>7.1f}% | {bp:>7.1f}%")

print(f"\n  All-wrong questions (0 of 5 correct):")
print(f"    Guided: {all_wrong_guided}  |  Baseline: {all_wrong_baseline}  |  Delta: {all_wrong_baseline - all_wrong_guided} fewer with guidance")

a2_data = {
    "dataset": "CommonsenseQA",
    "guided_mean_consistency": round(g_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "guided_wins": guided_wins,
    "baseline_wins": baseline_wins,
    "tied": tied,
    "guided_distribution": g_dist,
    "baseline_distribution": b_dist,
    "all_wrong_guided": all_wrong_guided,
    "all_wrong_baseline": all_wrong_baseline,
    "guided_letter_dist": {k: round(v/max(total_g_letters,1)*100,1) for k,v in g_letter_dist.items()},
    "baseline_letter_dist": {k: round(v/max(total_b_letters,1)*100,1) for k,v in b_letter_dist.items()},
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(a2_data, f, indent=2)
print(f"\nAngle 2 saved to {CONFIG['angle2_file']}")


In [ ]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
# =================================================================
# Confidence = fraction of votes that agreed on winning answer.
# ECE = average |accuracy - confidence| weighted by bucket size.
# Lower ECE = better calibrated.
# Random chance is 20% for 5-option MCQ.
# =================================================================

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})

    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    mc     = [r for r in results if 0.60 <= r["confidence"] < 0.80]
    mc_acc = sum(r["correct"] for r in mc) / max(1, len(mc)) * 100
    print(f"  {'ECE':<26}   {ece:.4f}")
    print(f"  High-conf : {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    print(f"  Mid-conf  : {len(mc)} questions  |  Accuracy: {mc_acc:.1f}%")
    return ece, calib_out, false_conf, hc_acc, len(hc), mc_acc, len(mc)

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (CommonsenseQA)")
print("=" * 65)
g_ece, g_calib, g_fc, g_hc_acc, g_hc_n, g_mc_acc, g_mc_n = calibration_report(all_results,  "GUIDED")
b_ece, b_calib, b_fc, b_hc_acc, b_hc_n, b_mc_acc, b_mc_n = calibration_report(base_results, "BASELINE")

ece_improvement = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE improvement : {ece_improvement:.1f}% better calibrated with guidance")
print(f"  ECE threshold   : Guided {'PASSES' if g_ece < 0.10 else 'FAILS'} the <0.10 threshold")
print(f"  False confidence: Guided {g_fc}  vs  Baseline {b_fc}  ({b_fc - g_fc} fewer with guidance)")

a3_data = {
    "dataset": "CommonsenseQA",
    "guided_ece": round(g_ece, 4),
    "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(ece_improvement, 1),
    "guided_calibration": g_calib,
    "baseline_calibration": b_calib,
    "guided_false_confidence": g_fc,
    "baseline_false_confidence": b_fc,
    "guided_high_conf_accuracy": round(g_hc_acc, 1),
    "baseline_high_conf_accuracy": round(b_hc_acc, 1),
    "guided_high_conf_n": g_hc_n,
    "baseline_high_conf_n": b_hc_n,
    "guided_mid_conf_accuracy": round(g_mc_acc, 1),
    "baseline_mid_conf_accuracy": round(b_mc_acc, 1),
    "guided_mid_conf_n": g_mc_n,
    "baseline_mid_conf_n": b_mc_n,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(a3_data, f, indent=2)
print(f"\nAngle 3 saved to {CONFIG['angle3_file']}")


In [ ]:
# CELL 16 -- Full Paper Summary (all three angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n = a1["n_questions"]

print("=" * 68)
print("  COMMONSENSEQA EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset: CommonsenseQA  |  Split: validation  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print(f"  Random chance baseline: 20.0%  (5 options, A-E)")
print(f"  NOTE: Guide fine-tuned on math (GSM8K) -- out-of-domain test")
print()

rows = [
    ["Metric",                   "Baseline",                                    "Guided",                                      "Change"],
    ["Overall Accuracy",
     str(a1["baseline_accuracy"]) + "%",
     str(a1["guided_accuracy"]) + "%",
     "+" + str(round(a1["guided_accuracy"]-a1["baseline_accuracy"],1)) + " pts"],
    ["Above Random Chance (20%)",
     "+" + str(a1["baseline_above_chance"]) + " pts",
     "+" + str(a1["guided_above_chance"]) + " pts", ""],
    ["Compute Cost",
     str(a1["baseline_compute_B"]) + "B passes",
     str(a1["guided_compute_B"]) + "B passes",
     str(a1["compute_savings_pct"]) + "% cheaper than ceiling"],
    ["Wasted Votes",
     str(a1["baseline_wasted_votes"]),
     str(a1["guided_wasted_votes"]),
     str(a1["wasted_votes_saved"]) + " fewer"],
    ["Vote Consistency",
     str(round(a2["baseline_mean_consistency"]*100,1)) + "%",
     str(round(a2["guided_mean_consistency"]*100,1)) + "%",
     str(round(a2["consistency_lift"],2)) + "x lift"],
    ["High-Agreement (80-100%)",
     str(a2["baseline_distribution"]["high   (80-100%)"]),
     str(a2["guided_distribution"]["high   (80-100%)"]), ""],
    ["All-Wrong (0/5 correct)",
     str(a2["all_wrong_baseline"]),
     str(a2["all_wrong_guided"]),
     str(a2["all_wrong_baseline"] - a2["all_wrong_guided"]) + " fewer complete failures"],
    ["ECE (lower = better)",
     str(a3["baseline_ece"]),
     str(a3["guided_ece"]),
     str(a3["ece_improvement_pct"]) + "% better"],
    ["High-Conf Accuracy (>=0.80)",
     str(a3["baseline_high_conf_accuracy"]) + "% (n=" + str(a3["baseline_high_conf_n"]) + ")",
     str(a3["guided_high_conf_accuracy"])  + "% (n=" + str(a3["guided_high_conf_n"])  + ")", ""],
    ["False Confidence Count",
     str(a3["baseline_false_confidence"]),
     str(a3["guided_false_confidence"]),
     str(a3["baseline_false_confidence"] - a3["guided_false_confidence"]) + " fewer"],
]

col_w = [32, 26, 26, 30]
header = rows[0]
sep    = "  " + "-+-".join("-" * w for w in col_w)
print("  " + " | ".join(f"{h:<{col_w[i]}}" for i, h in enumerate(header)))
print(sep)
for row in rows[1:]:
    print("  " + " | ".join(f"{str(row[i]):<{col_w[i]}}" for i in range(len(col_w))))

print()
print("=" * 68)
print("  KEY FINDING:")
print(f"  Guided pipeline achieves +{a1['accuracy_gain']:.1f} pts on CommonsenseQA")
print(f"  (out-of-domain: guide trained on math, tested on commonsense).")
print(f"  Calibration improves {a3['ece_improvement_pct']:.1f}% (ECE: {a3['baseline_ece']} -> {a3['guided_ece']}).")
print(f"  This confirms pipeline benefit is NOT math-specific.")
print("=" * 68)


### Part 05

# Notebook 12 -- Three-Angle Evaluation on RACE
## SLM-to-SLM Guided Reasoning Pipeline

**Why RACE?**

RACE (ReAding Comprehension from Examinations) contains English reading
comprehension questions from Chinese middle and high school exams.
Each question comes with a passage, and answering correctly requires:
- Understanding the passage content
- Inferring information not stated explicitly
- Applying logical reasoning across multiple sentences

This is fundamentally different from all previous datasets:
- Unlike math: no numeric computation
- Unlike science: requires reading a passage, not recalling facts
- Unlike commonsense: answer must be grounded in the given text

The guide must produce a plan that identifies WHICH part of the passage
is relevant and WHAT inference is needed — testing a new kind of reasoning.

**Pipeline Under Test:**
- Guide  : Qwen 2.5-3B-Instruct + LoRA fine-tuned adapter (1 forward pass)
- Solver : Qwen 2.5-1.5B-Instruct × 5 majority-vote passes
- Baseline: Qwen 2.5-1.5B-Instruct × 5 majority-vote passes (no guide plan)
- Random chance: **25.0%** (4 options: A, B, C, D)

**Dataset config:** We use the "high" subset (high school level) for
harder, more inference-heavy questions. The "middle" subset is easier
and more fact-retrieval focused.

**Three Evaluation Angles:**
1. Compute Efficiency  -- accuracy per billion parameter-passes
2. Vote Consistency    -- how reliably the ensemble agrees on the correct answer
3. Confidence Calibration -- how well confidence predicts correctness (ECE)


In [ ]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "ehovy/race",
    "dataset_config"      : "high",       # high school level -- harder, more inference-heavy
    "dataset_split"       : "test",
    "max_eval_samples"    : 500,
    "random_seed"         : 42,           # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 512,          # more tokens -- passage + question is longer

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


In [ ]:
# CELL 5 -- Load RACE dataset
# RACE fields:
#   example_id  : str
#   article     : str  (the reading passage)
#   answer      : str  (single letter: 'A', 'B', 'C', or 'D')
#   question    : str
#   options     : list of 4 strings (the answer choices, already plain text)
#
# We format the input as:
#   Passage: <article>
#   Question: <question>
#   Options:
#   A) ...
#   B) ...
#   C) ...
#   D) ...
#
# NOTE: passages can be long. We truncate to 600 chars to stay within
# model context limits while preserving enough for inference.

PASSAGE_MAX_CHARS = 600  # truncate long passages for context window safety

print("Loading RACE (high) from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_config"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Test size: {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record keys: {list(ex.keys())}")
print(f"Answer: {ex['answer']}")
print(f"Question: {ex['question'][:100]}")
print(f"Options: {ex['options']}")

VALID_LETTERS = set("ABCD")

def normalise_race(item):
    """Convert RACE record to {question, answer} pipeline format."""
    article  = item["article"].strip()
    # Truncate very long passages
    if len(article) > PASSAGE_MAX_CHARS:
        article = article[:PASSAGE_MAX_CHARS].rsplit(" ", 1)[0] + "..."

    opts = item["options"]   # list of 4 strings
    options_str = "\n".join(f"{chr(65+i)}) {opt}" for i, opt in enumerate(opts))

    q = (
        f"Passage:\n{article}\n\n"
        f"Question: {item['question'].strip()}\n\n"
        f"Options:\n{options_str}"
    )
    ans = str(item["answer"]).strip().upper()
    # Occasionally answer is '1','2','3','4' -- normalise
    num_map = {"1":"A","2":"B","3":"C","4":"D"}
    ans = num_map.get(ans, ans)
    return {"question": q, "answer": ans}


all_data = [normalise_race(x) for x in raw_ds[CONFIG["dataset_split"]]]

# CRITICAL: fix seed ONCE before sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

print(f"\nSample formatted input (first 400 chars):\n{test_data[0]['question'][:400]}")
print(f"\nAnswer: {test_data[0]['answer']}")


In [ ]:
# CELL 6 -- Answer extraction for multiple-choice (A-D)
# RACE answers are single letters A, B, C, or D.

def extract_gt_answer(answer_str):
    """GT is already a clean letter."""
    s = str(answer_str).strip().upper()
    num_map = {"1":"A","2":"B","3":"C","4":"D"}
    s = num_map.get(s, s)
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    """
    Extract chosen letter (A-D) from model free-form output.
    Priority: explicit answer phrases first, fallback to last standalone letter.
    """
    text = text.strip()

    # 1. Explicit answer phrases
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is)"
        r"[\s:]*([A-D])\b",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 2. "option/choice X is correct"
    m = re.search(
        r"(?:option|choice)\s+([A-D])\s+(?:is correct|is the answer|is right|matches)",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 3. #### A marker
    m = re.search(r"####\s*([A-D])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 4. Parenthesised at end: (A)
    m = re.search(r"\(([A-D])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 5. Bold: **A**
    m = re.search(r"\*\*([A-D])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 6. Standalone letter on its own line
    matches = re.findall(r"^\s*([A-D])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    # 7. Last standalone letter anywhere
    matches = re.findall(r"\b([A-D])\b", text, re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    return ""

# Quick test
for t in ["The answer is C", "#### B", "(A)", "**D**", "answer: B"]:
    print(f"  '{t}' -> '{extract_pred_answer(t)}'")
print("Extraction OK")


In [ ]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        ADAPTER_PATH,
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None

print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token



if 'guide_model' in dir():
    del guide_model
if 'adapter_path' in dir():
    del adapter_path
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Loading guide base: {CONFIG['guide_base']}")







guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


In [ ]:
# CELL 9 -- Prompts and generation functions
# RACE-specific: guide reads the passage and identifies:
#   1. Which part of the passage is relevant to the question
#   2. What inference or reasoning connects passage to answer
#   3. Which option matches that reasoning

GUIDE_SYSTEM = (
    "You are a reading comprehension assistant for multiple-choice questions.\n"
    "You are given a passage and a question with options A-D.\n"
    "Write 2-3 concrete reasoning steps that lead to the correct answer.\n"
    "Each step must reference SPECIFIC text from the passage.\n"
    "Your LAST line must always be: Best answer: <letter> because <one-line reason>\n\n"
    "Rules:\n"
    "- Quote or paraphrase the EXACT part of the passage that supports the answer.\n"
    "- Explicitly eliminate at least one wrong option if possible.\n"
    "- Do not guess -- every step must be grounded in the passage text.\n"
    "- No markdown. Plain text only.\n\n"
    "BAD example (not grounded):\n"
    "  Step 1: The passage talks about the topic.\n"
    "  Step 2: Option C seems most relevant.\n"
    "  Best answer: C because it sounds right\n\n"
    "GOOD example (grounded in passage):\n"
    "  Step 1: The passage says 'the factory closed in 1987 due to falling demand'.\n"
    "           This directly answers why the factory closed -- economic reasons.\n"
    "  Step 2: Option A says 'fire' and option B says 'flood' -- neither is in the passage.\n"
    "           Option D says 'falling demand' -- matches the passage exactly.\n"
    "  Best answer: D because the passage explicitly states 'falling demand' as the cause\n\n"
    "Apply this pattern to any reading comprehension question -- inference, main idea, "
    "vocabulary in context, author purpose, or detail retrieval."
)

SOLVE_SYSTEM = (
    "You are a precise reading comprehension solver.\n"
    "You are given a passage, a question, and a reasoning plan.\n"
    "Follow the plan exactly. Pick the letter the plan identifies as correct.\n"
    "Do not contradict the plan or re-read the passage independently.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]\n\n"
    "Example:\n"
    "Plan says: Best answer: D because the passage explicitly states 'falling demand'.\n"
    "The answer is D"
)

SOLVE_BASELINE_SYSTEM = (
    "You are a reading comprehension question answering assistant.\n"
    "Read the passage carefully. Answer the question based only on the passage.\n"
    "Think step by step if needed.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful reading comprehension checker.\n"
    "You are given a passage+question and a list of tied candidate answers.\n"
    "Re-read the passage and determine which answer is best supported by the text.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)


def _generate(model, tokenizer, system_prompt, user_prompt, temperature, max_new_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def generate_plan(question):
    return _generate(
        guide_model, guide_tok,
        GUIDE_SYSTEM, question,
        CONFIG["guide_temperature"], CONFIG["max_new_tokens"]
    )

def generate_guided(question, plan):
    prompt = f"Plan:\n{plan}\n\nNow answer:\n{question}"
    return _generate(
        resp_model, resp_tok,
        SOLVE_SYSTEM, prompt,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_baseline(question):
    return _generate(
        resp_model, resp_tok,
        SOLVE_BASELINE_SYSTEM, question,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_refiner(question, tied_answers):
    prompt = (
        f"Passage + Question:\n{question}\n\n"
        f"Tied candidate answers: {', '.join(tied_answers)}\n"
        f"Which is best supported by the passage?"
    )
    return _generate(
        resp_model, resp_tok,
        REFINER_SYSTEM, prompt,
        CONFIG["refiner_temperature"], CONFIG["max_new_tokens"]
    )

print("Prompts and generation functions ready")


In [ ]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, gt_answer=None):
    """Majority voting with refiner fallback on ties."""
    valid = [a for a in answers if a in VALID_LETTERS]
    if not valid:
        valid = answers

    vote_counts  = Counter(valid)
    most_common  = vote_counts.most_common()
    top_answer   = most_common[0][0]
    top_count    = most_common[0][1]
    total        = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans in VALID_LETTERS else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted     = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "total_votes"      : total,
        "correct_votes"    : correct_votes,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }

print("Voting logic ready")


In [ ]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (RACE-High)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Input (first 500 chars):\n{q[:500]}")
print(f"\nGT Answer: {gt}")

print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")

print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline: {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print(f"  Vote counts: {b['vote_counts']}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


In [ ]:
# CELL 12 -- Full Dual Evaluation Loop

print(f"Dual evaluation: {len(test_data)} RACE-High questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print(f"Random baseline (chance): 25.0% (1 in 4 options)")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="RACE Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "plan"             : plan,
            "votes"            : g_votes_raw,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "vote_counts"      : g_dec["vote_counts"],
            "total_votes"      : g_dec["total_votes"],
            "correct_votes"    : g_dec["correct_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [GUIDED ERROR idx={idx}]: {e}")
        all_results.append({"mode":"guided","idx":idx,"correct":False,
                             "gt_answer":gt_answer,"final_answer":"",
                             "strategy":"error","confidence":0.0,
                             "vote_consistency":0.0,"wasted_votes":5,
                             "refiner_used":False,"refiner_correct":None,
                             "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "votes"            : b_votes_raw,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "vote_counts"      : b_dec["vote_counts"],
            "total_votes"      : b_dec["total_votes"],
            "correct_votes"    : b_dec["correct_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : b_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [BASELINE ERROR idx={idx}]: {e}")
        base_results.append({"mode":"baseline","idx":idx,"correct":False,
                              "gt_answer":gt_answer,"final_answer":"",
                              "strategy":"error","confidence":0.0,
                              "vote_consistency":0.0,"wasted_votes":5,
                              "refiner_used":False,"refiner_correct":None,
                              "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- Checkpoint -------------------------------------------
    if (idx + 1) % CONFIG["save_every"] == 0 or (idx + 1) == len(test_data):
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        elapsed = time.time() - t0
        g_acc_so_far = sum(r["correct"] for r in all_results) / len(all_results) * 100
        b_acc_so_far = sum(r["correct"] for r in base_results) / len(base_results) * 100
        print(f"  [{idx+1}/{len(test_data)}] Guided: {g_acc_so_far:.1f}%  "
              f"Baseline: {b_acc_so_far:.1f}%  ({elapsed/60:.1f}min)")

print("\n" + "=" * 65)
g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
print(f"  Guided   : {g_acc:.1f}%")
print(f"  Baseline : {b_acc:.1f}%")
print(f"  Delta    : +{g_acc - b_acc:.1f} pts")
print(f"  Random   : 25.0%")
print("=" * 65)


In [ ]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY

G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]
guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N
random_chance    = 25.0

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted = sum(r["wasted_votes"] for r in all_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (RACE-High)")
print("=" * 65)
print(f"  Random chance: {random_chance}% (4 options)")
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}")
print(f"  {'Random chance':<32} | {'--':>10} | {random_chance:>8.1f}%")
print(f"  {'Baseline (1.5B x5)':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}%")
print(f"  {'Guided  (3B x1 + 1.5B x5)':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}%")
print(f"  {'Upper   (3B x5)':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9}")
print(f"\n  Gain            : +{g_acc - b_acc:.1f} pts")
print(f"  Above chance    : +{g_acc - random_chance:.1f} pts")
print(f"  Compute savings : {savings_pct:.0f}% cheaper vs upper")
print(f"  Wasted saved    : {b_wasted - g_wasted}")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")

print("\n  Strategy breakdown:")
for strat, stats in strategy_stats.items():
    acc = stats["correct"] / stats["n"] * 100 if stats["n"] else 0
    print(f"    {strat:<20}: {stats['n']}q  |  {acc:.1f}% accurate")

a1_data = {
    "dataset": "RACE-High",
    "n_questions": len(all_results),
    "random_chance": random_chance,
    "guided_accuracy": round(g_acc, 1),
    "baseline_accuracy": round(b_acc, 1),
    "accuracy_gain": round(g_acc - b_acc, 1),
    "guided_above_chance": round(g_acc - random_chance, 1),
    "baseline_above_chance": round(b_acc - random_chance, 1),
    "guided_compute_B": guided_compute,
    "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute,
    "compute_savings_pct": round(savings_pct, 1),
    "guided_wasted_votes": g_wasted,
    "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted - g_wasted,
    "refiner_triggered": ref_triggered,
    "refiner_correct": ref_correct,
    "strategy_stats": strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(a1_data, f, indent=2)
print(f"\nAngle 1 saved to {CONFIG['angle1_file']}")


In [ ]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

all_letters_guided   = []
all_letters_baseline = []
for r in all_results:
    all_letters_guided.extend(r["vote_counts"].keys())
for r in base_results:
    all_letters_baseline.extend(r["vote_counts"].keys())
g_letter_dist = Counter(all_letters_guided)
b_letter_dist = Counter(all_letters_baseline)
total_g = sum(g_letter_dist.values())
total_b = sum(b_letter_dist.values())

all_wrong_guided   = sum(1 for r in all_results  if r["vote_consistency"] == 0.0)
all_wrong_baseline = sum(1 for r in base_results if r["vote_consistency"] == 0.0)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (RACE-High)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio:")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} / {CONFIG['n_votes']} votes correct avg)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} / {CONFIG['n_votes']} votes correct avg)")
print(f"    Lift     : {lift:.2f}x")
print(f"\n  Per-question: Guided wins {guided_wins}, Baseline wins {baseline_wins}, Tied {tied}")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gn, bn = g_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {gn:>8} | {bn:>8} | {gn-bn:>+6}")

print(f"\n  Letter distribution (% of winning votes):")
print(f"  {'Letter':<8} | {'Guided':>8} | {'Baseline':>8}")
for ltr in sorted(VALID_LETTERS):
    gp = g_letter_dist.get(ltr, 0) / max(total_g, 1) * 100
    bp = b_letter_dist.get(ltr, 0) / max(total_b, 1) * 100
    print(f"  {ltr:<8} | {gp:>7.1f}% | {bp:>7.1f}%")

print(f"\n  All-wrong: Guided {all_wrong_guided}  |  Baseline {all_wrong_baseline}  |  Delta {all_wrong_baseline - all_wrong_guided} fewer")

a2_data = {
    "dataset": "RACE-High",
    "guided_mean_consistency": round(g_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "guided_wins": guided_wins,
    "baseline_wins": baseline_wins,
    "tied": tied,
    "guided_distribution": g_dist,
    "baseline_distribution": b_dist,
    "all_wrong_guided": all_wrong_guided,
    "all_wrong_baseline": all_wrong_baseline,
    "guided_letter_dist": {k: round(v/max(total_g,1)*100,1) for k,v in g_letter_dist.items()},
    "baseline_letter_dist": {k: round(v/max(total_b,1)*100,1) for k,v in b_letter_dist.items()},
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(a2_data, f, indent=2)
print(f"\nAngle 2 saved to {CONFIG['angle2_file']}")


In [ ]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})

    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"  ECE: {ece:.4f}")
    print(f"  High-conf: {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    return ece, calib_out, false_conf, hc_acc, len(hc)

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (RACE-High)")
print("=" * 65)
g_ece, g_calib, g_fc, g_hc_acc, g_hc_n = calibration_report(all_results,  "GUIDED")
b_ece, b_calib, b_fc, b_hc_acc, b_hc_n = calibration_report(base_results, "BASELINE")

ece_imp = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE improvement : {ece_imp:.1f}% better calibrated with guidance")
print(f"  False confidence: Guided {g_fc}  vs  Baseline {b_fc}  ({b_fc - g_fc} fewer)")

a3_data = {
    "dataset": "RACE-High",
    "guided_ece": round(g_ece, 4),
    "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(ece_imp, 1),
    "guided_calibration": g_calib,
    "baseline_calibration": b_calib,
    "guided_false_confidence": g_fc,
    "baseline_false_confidence": b_fc,
    "guided_high_conf_accuracy": round(g_hc_acc, 1),
    "baseline_high_conf_accuracy": round(b_hc_acc, 1),
    "guided_high_conf_n": g_hc_n,
    "baseline_high_conf_n": b_hc_n,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(a3_data, f, indent=2)
print(f"\nAngle 3 saved to {CONFIG['angle3_file']}")


In [ ]:
# CELL 16 -- Full Paper Summary

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n = a1["n_questions"]
print("=" * 68)
print("  RACE-HIGH EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset: RACE (high school)  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print(f"  Random chance: 25.0%  (4 options)")
print()

rows = [
    ["Metric", "Baseline", "Guided", "Change"],
    ["Overall Accuracy", str(a1["baseline_accuracy"])+"%", str(a1["guided_accuracy"])+"%",
     "+"+str(round(a1["guided_accuracy"]-a1["baseline_accuracy"],1))+" pts"],
    ["Above Random (25%)", "+"+str(a1["baseline_above_chance"])+" pts",
     "+"+str(a1["guided_above_chance"])+" pts", ""],
    ["Compute Cost", str(a1["baseline_compute_B"])+"B passes",
     str(a1["guided_compute_B"])+"B passes", str(a1["compute_savings_pct"])+"% cheaper vs ceiling"],
    ["Wasted Votes", str(a1["baseline_wasted_votes"]), str(a1["guided_wasted_votes"]),
     str(a1["wasted_votes_saved"])+" fewer"],
    ["Vote Consistency", str(round(a2["baseline_mean_consistency"]*100,1))+"%",
     str(round(a2["guided_mean_consistency"]*100,1))+"%", str(round(a2["consistency_lift"],2))+"x lift"],
    ["High-Agreement (80-100%)", str(a2["baseline_distribution"]["high   (80-100%)"]),
     str(a2["guided_distribution"]["high   (80-100%)"]), ""],
    ["All-Wrong (0/5)", str(a2["all_wrong_baseline"]), str(a2["all_wrong_guided"]),
     str(a2["all_wrong_baseline"]-a2["all_wrong_guided"])+" fewer"],
    ["ECE (lower=better)", str(a3["baseline_ece"]), str(a3["guided_ece"]),
     str(a3["ece_improvement_pct"])+"% better"],
    ["High-Conf Accuracy (>=0.80)",
     str(a3["baseline_high_conf_accuracy"])+"% (n="+str(a3["baseline_high_conf_n"])+")",
     str(a3["guided_high_conf_accuracy"])+"% (n="+str(a3["guided_high_conf_n"])+")", ""],
    ["False Confidence", str(a3["baseline_false_confidence"]), str(a3["guided_false_confidence"]),
     str(a3["baseline_false_confidence"]-a3["guided_false_confidence"])+" fewer"],
]

col_w = [32, 26, 26, 30]
print("  "+" | ".join(f"{rows[0][i]:<{col_w[i]}}" for i in range(4)))
print("  "+"+-".join("-"*w for w in col_w))
for row in rows[1:]:
    print("  "+" | ".join(f"{str(row[i]):<{col_w[i]}}" for i in range(4)))

print()
print(f"  KEY FINDING: +{a1['accuracy_gain']:.1f} pts on reading comprehension + inference")
print(f"  at 30% lower compute than ceiling. ECE improves {a3['ece_improvement_pct']:.1f}%.")
print("=" * 68)


### PArt 06

# Notebook 13 -- Three-Angle Evaluation on PIQA
## SLM-to-SLM Guided Reasoning Pipeline

**Why PIQA?**

PIQA (Physical Intuition Question Answering) tests physical and procedural
reasoning about the everyday world. Questions take the form:
  "How do you accomplish goal G?"
  Solution 1: <plausible or implausible method>
  Solution 2: <plausible or implausible method>

The model must pick the more physically correct or practical solution.

This adds two things no previous dataset provides:
1. **A new reasoning domain** -- physical world knowledge, object properties,
   cause-and-effect of physical actions
2. **A new answer format** -- binary choice (2 options) instead of 4 or 5.
   Random chance is 50%, so any gain above 50% is meaningful.

The guide (fine-tuned on math GSM8K) must now reason about physical
plausibility -- the maximum possible domain gap test alongside CommonsenseQA.

**Pipeline Under Test:**
- Guide  : Qwen 2.5-3B-Instruct + LoRA fine-tuned adapter (1 forward pass)
- Solver : Qwen 2.5-1.5B-Instruct x 5 majority-vote passes
- Baseline: Qwen 2.5-1.5B-Instruct x 5 majority-vote passes (no guide plan)
- Random chance: **50.0%** (binary: solution 1 or solution 2)

**Note on answer labels:**
PIQA uses numeric labels: 0 = Solution 1, 1 = Solution 2.
We map these to 'A' and 'B' for consistency with the pipeline.

**Three Evaluation Angles:**
1. Compute Efficiency  -- accuracy per billion parameter-passes
2. Vote Consistency    -- how reliably the ensemble agrees on the correct answer
3. Confidence Calibration -- how well confidence predicts correctness (ECE)


In [ ]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "nthngdy/piqa",
    "dataset_split"       : "validation",  # test labels are withheld; validation has labels
    "max_eval_samples"    : 900,
    "random_seed"         : 42,            # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 300,           # binary choice -- shorter answers needed

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


In [ ]:
# CELL 5 -- Load PIQA dataset
# PIQA fields:
#   goal     : str  (what the person wants to accomplish)
#   sol1     : str  (solution 1)
#   sol2     : str  (solution 2)
#   label    : int  (0 = sol1 is correct, 1 = sol2 is correct)
#
# We map label 0 -> 'A', label 1 -> 'B' for pipeline consistency.
# Format:
#   Goal: <goal>
#   A) <sol1>
#   B) <sol2>

print("Loading PIQA from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Val size : {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record:")
for k, v in ex.items():
    print(f"  {k}: {v}")

VALID_LETTERS = set("AB")
LABEL_TO_LETTER = {0: "A", 1: "B"}

def normalise_piqa(item):
    """Convert PIQA record to {question, answer} pipeline format."""
    q = (
        f"Goal: {item['goal'].strip()}\n\n"
        f"Which solution is more physically correct or practical?\n\n"
        f"A) {item['sol1'].strip()}\n"
        f"B) {item['sol2'].strip()}"
    )
    ans = LABEL_TO_LETTER[int(item["label"])]
    return {"question": q, "answer": ans,
            "goal": item["goal"], "sol1": item["sol1"], "sol2": item["sol2"]}


all_data = [normalise_piqa(x) for x in raw_ds[CONFIG["dataset_split"]]]

# CRITICAL: fix seed ONCE before sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

# Check label balance in sample
a_count = sum(1 for x in test_data if x["answer"] == "A")
b_count = sum(1 for x in test_data if x["answer"] == "B")
print(f"\nLabel balance: A={a_count}  B={b_count}  (ideal: 50/50)")
print(f"\nSample question:\n{test_data[0]['question']}")
print(f"Answer: {test_data[0]['answer']}")


In [ ]:
# CELL 6 -- Answer extraction for binary choice (A or B)
# PIQA answers are A or B only.
# We need a tight extractor that doesn't false-fire on
# mid-sentence A/B mentions like "method A is faster".

def extract_gt_answer(answer_str):
    """GT is already 'A' or 'B'."""
    s = str(answer_str).strip().upper()
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    """
    Extract A or B from model free-form output.
    Priority: explicit conclusive phrases first, fallback to last letter.
    Tight patterns to avoid false-fires on 'method A' / 'solution B' mid-sentence.
    """
    text = text.strip()

    # 1. Explicit answer phrases
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is|"
        r"best solution is|better solution is|solution is)"
        r"[\s:]*([AB])\b",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 2. "Solution/Option A is correct/better/more practical"
    m = re.search(
        r"(?:solution|option)\s+([AB])\s+(?:is correct|is better|is more|is the|works better|is practical)",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 3. #### A marker
    m = re.search(r"####\s*([AB])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 4. Parenthesised at end: (A) or (B)
    m = re.search(r"\(([AB])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 5. Bold: **A** or **B**
    m = re.search(r"\*\*([AB])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 6. Standalone letter on its own line (last occurrence)
    matches = re.findall(r"^\s*([AB])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    # 7. "I would choose A/B" or "I choose A/B"
    m = re.search(r"(?:i would choose|i choose|choose)\s+([AB])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 8. Last standalone A or B (loose fallback -- only fires if nothing else matched)
    matches = re.findall(r"\b([AB])\b", text, re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    return ""

# Quick test
test_outputs = [
    "After thinking about it, the answer is A",
    "Solution B is more practical for this task",
    "#### B",
    "(A)",
    "**B**",
    "I would choose A because it uses less force",
]
for t in test_outputs:
    print(f"  '{t[:60]}' -> '{extract_pred_answer(t)}'")
print("Extraction OK")


In [ ]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        ADAPTER_PATH,
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None

print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token







if 'guide_model' in dir():
    del guide_model
if 'adapter_path' in dir():
    del adapter_path
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Loading guide base: {CONFIG['guide_base']}")








guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


In [ ]:
# CELL 9 -- Prompts and generation functions
# PIQA-specific: guide reasons about physical plausibility.
# The guide must evaluate WHICH solution is more physically correct,
# safe, practical, or achieves the goal effectively.
# No math. No passage. Pure physical world knowledge.

GUIDE_SYSTEM = (
    "You are a physical reasoning assistant for binary-choice questions.\n"
    "You are given a goal and two solutions (A and B).\n"
    "Write 2-3 concrete reasoning steps about which solution is more physically correct.\n"
    "Your LAST line must always be: Best answer: <A or B> because <one-line physical reason>\n\n"
    "Rules:\n"
    "- Reason about physical properties: material, force, safety, practicality, cause-effect.\n"
    "- Explicitly explain why the OTHER solution fails or is less practical.\n"
    "- Be specific -- reference the actual objects and actions in the question.\n"
    "- No markdown. Plain text only.\n\n"
    "BAD example (too vague):\n"
    "  Step 1: Think about which solution makes more sense.\n"
    "  Step 2: A seems better.\n"
    "  Best answer: A because it is correct\n\n"
    "GOOD example (physically grounded):\n"
    "  Goal: How to remove a stripped screw.\n"
    "  A) Use a rubber band between the screwdriver and screw for extra grip.\n"
    "  B) Pour water on the screw to make it easier to turn.\n"
    "  Step 1: A stripped screw has no grooves for the screwdriver to grip.\n"
    "           Adding a rubber band increases friction between driver and screw head.\n"
    "  Step 2: Water does not restore grip on a stripped screw -- it may cause rust.\n"
    "           Solution B does not address the core problem (lack of grip).\n"
    "  Best answer: A because rubber band friction compensates for missing screw grooves\n\n"
    "Apply this pattern to any physical task -- cooking, cleaning, building, tools, or materials."
)

SOLVE_SYSTEM = (
    "You are a precise physical reasoning solver.\n"
    "You are given a goal with two solutions (A and B) and a reasoning plan.\n"
    "Follow the plan exactly. Pick the letter (A or B) the plan identifies as correct.\n"
    "Do not contradict the plan.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [A or B]\n\n"
    "Example:\n"
    "Plan says: Best answer: A because rubber band friction compensates for missing grooves.\n"
    "The answer is A"
)

SOLVE_BASELINE_SYSTEM = (
    "You are a physical reasoning assistant.\n"
    "You are given a goal and two solutions (A and B).\n"
    "Decide which solution is more physically correct, practical, or effective.\n"
    "Think step by step if needed.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [A or B]"
)

REFINER_SYSTEM = (
    "You are a careful physical reasoning checker.\n"
    "You are given a goal with two solutions and a tie between A and B.\n"
    "Reason about which solution is more physically sound or practical.\n"
    "Your absolute last line must be exactly: The answer is [A or B]"
)


def _generate(model, tokenizer, system_prompt, user_prompt, temperature, max_new_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def generate_plan(question):
    return _generate(
        guide_model, guide_tok,
        GUIDE_SYSTEM, question,
        CONFIG["guide_temperature"], CONFIG["max_new_tokens"]
    )

def generate_guided(question, plan):
    prompt = f"Plan:\n{plan}\n\nNow answer:\n{question}"
    return _generate(
        resp_model, resp_tok,
        SOLVE_SYSTEM, prompt,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_baseline(question):
    return _generate(
        resp_model, resp_tok,
        SOLVE_BASELINE_SYSTEM, question,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_refiner(question, tied_answers):
    prompt = (
        f"Goal + Solutions:\n{question}\n\n"
        f"Tied candidates: {', '.join(tied_answers)}\n"
        f"Which is physically more correct?"
    )
    return _generate(
        resp_model, resp_tok,
        REFINER_SYSTEM, prompt,
        CONFIG["refiner_temperature"], CONFIG["max_new_tokens"]
    )

print("Prompts and generation functions ready")


In [ ]:
# CELL 10 -- Voting logic with richer metrics
# NOTE: With binary choice (A/B only), ties are more frequent than
# with 4/5-choice MCQ -- a 2-2 or 3-2 split is common.
# The refiner handles ties as usual.

def vote_and_decide(answers, question, gt_answer=None):
    """Majority voting with refiner fallback on ties."""
    valid = [a for a in answers if a in VALID_LETTERS]
    if not valid:
        valid = answers

    vote_counts  = Counter(valid)
    most_common  = vote_counts.most_common()
    top_answer   = most_common[0][0]
    top_count    = most_common[0][1]
    total        = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans in VALID_LETTERS else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted     = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "total_votes"      : total,
        "correct_votes"    : correct_votes,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }

print("Voting logic ready")


In [ ]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (PIQA)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question:\n{q}")
print(f"\nGT Answer: {gt}")

print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")
print(f"  Vote counts: {g['vote_counts']}")

print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw: {raw[:80]}")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline: {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print(f"  Vote counts: {b['vote_counts']}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


In [ ]:
# CELL 12 -- Full Dual Evaluation Loop

print(f"Dual evaluation: {len(test_data)} PIQA questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print(f"Random baseline (chance): 50.0% (binary: A or B)")
print(f"NOTE: With binary choice, ties (e.g. 2A-2B-1A) are more common.")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="PIQA Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "plan"             : plan,
            "votes"            : g_votes_raw,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "vote_counts"      : g_dec["vote_counts"],
            "total_votes"      : g_dec["total_votes"],
            "correct_votes"    : g_dec["correct_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [GUIDED ERROR idx={idx}]: {e}")
        all_results.append({"mode":"guided","idx":idx,"correct":False,
                             "gt_answer":gt_answer,"final_answer":"",
                             "strategy":"error","confidence":0.0,
                             "vote_consistency":0.0,"wasted_votes":5,
                             "refiner_used":False,"refiner_correct":None,
                             "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "votes"            : b_votes_raw,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "vote_counts"      : b_dec["vote_counts"],
            "total_votes"      : b_dec["total_votes"],
            "correct_votes"    : b_dec["correct_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : b_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [BASELINE ERROR idx={idx}]: {e}")
        base_results.append({"mode":"baseline","idx":idx,"correct":False,
                              "gt_answer":gt_answer,"final_answer":"",
                              "strategy":"error","confidence":0.0,
                              "vote_consistency":0.0,"wasted_votes":5,
                              "refiner_used":False,"refiner_correct":None,
                              "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- Checkpoint -------------------------------------------
    if (idx + 1) % CONFIG["save_every"] == 0 or (idx + 1) == len(test_data):
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        elapsed = time.time() - t0
        g_acc_so_far = sum(r["correct"] for r in all_results) / len(all_results) * 100
        b_acc_so_far = sum(r["correct"] for r in base_results) / len(base_results) * 100
        print(f"  [{idx+1}/{len(test_data)}] Guided: {g_acc_so_far:.1f}%  "
              f"Baseline: {b_acc_so_far:.1f}%  ({elapsed/60:.1f}min)")

print("\n" + "=" * 65)
g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
print(f"  Guided   : {g_acc:.1f}%")
print(f"  Baseline : {b_acc:.1f}%")
print(f"  Delta    : +{g_acc - b_acc:.1f} pts")
print(f"  Random   : 50.0%")
print("=" * 65)


In [ ]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
# NOTE: Random chance for PIQA is 50% (binary).
# Any accuracy above 50% shows the model has learned something.
# The compute comparison is identical to other datasets.

G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]
guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N
random_chance    = 50.0

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted = sum(r["wasted_votes"] for r in all_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (PIQA)")
print("=" * 65)
print(f"  Random chance: {random_chance}% (binary A/B)")
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}")
print(f"  {'Random chance':<32} | {'--':>10} | {random_chance:>8.1f}%")
print(f"  {'Baseline (1.5B x5)':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}%")
print(f"  {'Guided  (3B x1 + 1.5B x5)':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}%")
print(f"  {'Upper   (3B x5)':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9}")
print(f"\n  Gain                    : +{g_acc - b_acc:.1f} pts")
print(f"  Above random chance     : +{g_acc - random_chance:.1f} pts")
print(f"  Baseline above chance   : +{b_acc - random_chance:.1f} pts")
print(f"  Compute savings vs upper: {savings_pct:.0f}% cheaper")
print(f"  Wasted votes saved      : {b_wasted - g_wasted}")
if ref_triggered:
    pct = ref_correct/ref_triggered*100
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({pct:.1f}%)")
else:
    print(f"  Refiner: 0 triggered")

print("\n  Strategy breakdown:")
for strat, stats in strategy_stats.items():
    acc = stats["correct"] / stats["n"] * 100 if stats["n"] else 0
    print(f"    {strat:<20}: {stats['n']}q  |  {acc:.1f}% accurate")

a1_data = {
    "dataset": "PIQA",
    "n_questions": len(all_results),
    "random_chance": random_chance,
    "guided_accuracy": round(g_acc, 1),
    "baseline_accuracy": round(b_acc, 1),
    "accuracy_gain": round(g_acc - b_acc, 1),
    "guided_above_chance": round(g_acc - random_chance, 1),
    "baseline_above_chance": round(b_acc - random_chance, 1),
    "guided_compute_B": guided_compute,
    "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute,
    "compute_savings_pct": round(savings_pct, 1),
    "guided_wasted_votes": g_wasted,
    "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted - g_wasted,
    "refiner_triggered": ref_triggered,
    "refiner_correct": ref_correct,
    "strategy_stats": strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(a1_data, f, indent=2)
print(f"\nAngle 1 saved to {CONFIG['angle1_file']}")


In [ ]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
# NOTE: With binary choice, the consistency distribution will look different.
# A 5-0 split gives 1.0 consistency if correct, 0.0 if wrong.
# A 3-2 split gives 0.6 or 0.4 depending on which side is correct.
# Expect more questions in the medium bucket vs 4/5-choice datasets.

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

all_letters_guided   = []
all_letters_baseline = []
for r in all_results:
    all_letters_guided.extend(r["vote_counts"].keys())
for r in base_results:
    all_letters_baseline.extend(r["vote_counts"].keys())
g_letter_dist = Counter(all_letters_guided)
b_letter_dist = Counter(all_letters_baseline)
total_g = sum(g_letter_dist.values())
total_b = sum(b_letter_dist.values())

all_wrong_guided   = sum(1 for r in all_results  if r["vote_consistency"] == 0.0)
all_wrong_baseline = sum(1 for r in base_results if r["vote_consistency"] == 0.0)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (PIQA)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Lift     : {lift:.2f}x")
print(f"\n  Per-question: Guided wins {guided_wins}, Baseline wins {baseline_wins}, Tied {tied}")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gn, bn = g_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {gn:>8} | {bn:>8} | {gn-bn:>+6}")

print(f"\n  Label distribution (% of winning votes) -- should be ~50/50:")
print(f"  {'Letter':<8} | {'Guided':>8} | {'Baseline':>8}")
for ltr in sorted(VALID_LETTERS):
    gp = g_letter_dist.get(ltr, 0) / max(total_g, 1) * 100
    bp = b_letter_dist.get(ltr, 0) / max(total_b, 1) * 100
    print(f"  {ltr:<8} | {gp:>7.1f}% | {bp:>7.1f}%")

print(f"\n  All-wrong (0/5): Guided {all_wrong_guided}  |  Baseline {all_wrong_baseline}"
      f"  |  {all_wrong_baseline - all_wrong_guided} fewer with guidance")

a2_data = {
    "dataset": "PIQA",
    "guided_mean_consistency": round(g_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "guided_wins": guided_wins,
    "baseline_wins": baseline_wins,
    "tied": tied,
    "guided_distribution": g_dist,
    "baseline_distribution": b_dist,
    "all_wrong_guided": all_wrong_guided,
    "all_wrong_baseline": all_wrong_baseline,
    "guided_letter_dist": {k: round(v/max(total_g,1)*100,1) for k,v in g_letter_dist.items()},
    "baseline_letter_dist": {k: round(v/max(total_b,1)*100,1) for k,v in b_letter_dist.items()},
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(a2_data, f, indent=2)
print(f"\nAngle 2 saved to {CONFIG['angle2_file']}")


In [ ]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
# NOTE: For binary choice, the only confidence values possible from 5 votes are:
#   1.0  (5-0 unanimous)
#   0.8  (4-1 split)
#   0.6  (3-2 split -- ties go to refiner, so this may not appear as majority)
# After refiner (6 total votes): 0.67, 0.83, 1.0
# So almost all questions will land in the Very High (>=0.80) bucket.
# The Low (<0.40) bucket will likely be empty.
# ECE still meaningful -- it measures how well high confidence predicts accuracy.

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})

    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"  ECE: {ece:.4f}")
    print(f"  High-conf: {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    return ece, calib_out, false_conf, hc_acc, len(hc)

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (PIQA)")
print("=" * 65)
g_ece, g_calib, g_fc, g_hc_acc, g_hc_n = calibration_report(all_results,  "GUIDED")
b_ece, b_calib, b_fc, b_hc_acc, b_hc_n = calibration_report(base_results, "BASELINE")

ece_imp = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE improvement : {ece_imp:.1f}% better calibrated with guidance")
print(f"  False confidence: Guided {g_fc}  vs  Baseline {b_fc}  ({b_fc - g_fc} fewer)")

a3_data = {
    "dataset": "PIQA",
    "guided_ece": round(g_ece, 4),
    "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(ece_imp, 1),
    "guided_calibration": g_calib,
    "baseline_calibration": b_calib,
    "guided_false_confidence": g_fc,
    "baseline_false_confidence": b_fc,
    "guided_high_conf_accuracy": round(g_hc_acc, 1),
    "baseline_high_conf_accuracy": round(b_hc_acc, 1),
    "guided_high_conf_n": g_hc_n,
    "baseline_high_conf_n": b_hc_n,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(a3_data, f, indent=2)
print(f"\nAngle 3 saved to {CONFIG['angle3_file']}")


In [ ]:
# CELL 16 -- Full Paper Summary (all three angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n = a1["n_questions"]

print("=" * 68)
print("  PIQA EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset : PIQA  |  Split: validation  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models  : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print(f"  Format  : Binary choice (A or B)  |  Random chance: 50.0%")
print(f"  NOTE    : Guide fine-tuned on math -- out-of-domain physical reasoning test")
print()

rows = [
    ["Metric",                    "Baseline",                                   "Guided",                                     "Change"],
    ["Overall Accuracy",
     str(a1["baseline_accuracy"]) + "%",
     str(a1["guided_accuracy"]) + "%",
     "+" + str(round(a1["guided_accuracy"]-a1["baseline_accuracy"],1)) + " pts"],
    ["Above Random Chance (50%)",
     "+" + str(a1["baseline_above_chance"]) + " pts",
     "+" + str(a1["guided_above_chance"]) + " pts", ""],
    ["Compute Cost",
     str(a1["baseline_compute_B"]) + "B passes",
     str(a1["guided_compute_B"]) + "B passes",
     str(a1["compute_savings_pct"]) + "% cheaper vs ceiling"],
    ["Wasted Votes",
     str(a1["baseline_wasted_votes"]),
     str(a1["guided_wasted_votes"]),
     str(a1["wasted_votes_saved"]) + " fewer"],
    ["Refiner Triggered",
     "--",
     str(a1["refiner_triggered"]) + " of " + str(n),
     ""],
    ["Vote Consistency",
     str(round(a2["baseline_mean_consistency"]*100,1)) + "%",
     str(round(a2["guided_mean_consistency"]*100,1)) + "%",
     str(round(a2["consistency_lift"],2)) + "x lift"],
    ["High-Agreement (80-100%)",
     str(a2["baseline_distribution"]["high   (80-100%)"]),
     str(a2["guided_distribution"]["high   (80-100%)"]), ""],
    ["All-Wrong (0/5 correct)",
     str(a2["all_wrong_baseline"]),
     str(a2["all_wrong_guided"]),
     str(a2["all_wrong_baseline"] - a2["all_wrong_guided"]) + " fewer complete failures"],
    ["ECE (lower = better)",
     str(a3["baseline_ece"]),
     str(a3["guided_ece"]),
     str(a3["ece_improvement_pct"]) + "% better"],
    ["High-Conf Accuracy (>=0.80)",
     str(a3["baseline_high_conf_accuracy"]) + "% (n=" + str(a3["baseline_high_conf_n"]) + ")",
     str(a3["guided_high_conf_accuracy"])  + "% (n=" + str(a3["guided_high_conf_n"])  + ")", ""],
    ["False Confidence Count",
     str(a3["baseline_false_confidence"]),
     str(a3["guided_false_confidence"]),
     str(a3["baseline_false_confidence"] - a3["guided_false_confidence"]) + " fewer"],
]

col_w = [32, 26, 26, 30]
print("  " + " | ".join(f"{rows[0][i]:<{col_w[i]}}" for i in range(4)))
print("  " + "+-".join("-"*w for w in col_w))
for row in rows[1:]:
    print("  " + " | ".join(f"{str(row[i]):<{col_w[i]}}" for i in range(4)))

print()
print("=" * 68)
print("  CROSS-DATASET SUMMARY (all 7 datasets):")
print("=" * 68)
cross = [
    ["SVAMP",        "Math",        "+13 pts", "1.52x", "22.3%", "+67"],
    ["ASDiv",        "Math",        "+11 pts", "1.19x", "53.2%", "0"],
    ["AQUA-RAT",     "Algebra",     "+12 pts", "1.00x", "45.6%", "-4"],
    ["ARC-Challenge","Science",     "+7 pts",  "1.09x", "32.6%", "+7"],
    ["CommonsenseQA","Commonsense", "+9 pts",  "1.14x", "33.7%", "+31"],
    ["RACE-High",    "Reading",     "+9 pts",  "1.17x", "23.1%", "+18"],
    ["PIQA",         "Physical",
     "+" + str(a1["accuracy_gain"]) + " pts",
     str(round(a2["consistency_lift"],2)) + "x",
     str(a3["ece_improvement_pct"]) + "%",
     str(a1["wasted_votes_saved"])],
]
hdr = ["Dataset", "Domain", "Acc Gain", "Cons. Lift", "ECE Imp.", "Wasted Saved"]
cw  = [16, 14, 10, 12, 10, 14]
print("  " + " | ".join(f"{hdr[i]:<{cw[i]}}" for i in range(6)))
print("  " + "+-".join("-"*w for w in cw))
for row in cross:
    print("  " + " | ".join(f"{row[i]:<{cw[i]}}" for i in range(6)))
print()
print(f"  KEY FINDING: Pipeline delivers positive accuracy gains across")
print(f"  ALL 7 datasets spanning 5 reasoning domains and 3 answer formats.")
print(f"  ECE improves on every single dataset -- the most reliable benefit.")
print("=" * 68)
